# 参数高效微调（PEFT）方法对比讲解

本 Notebook 以 BERT + SST-2 情感分类任务为实验平台，逐一实现并对比以下六种主流 PEFT 方法：
**冻结特征提取（Frozen）→ 全量微调（Full FT）→ BitFit → P-Tuning → Prefix Tuning → P-Tuning v2 → LoRA → Adapter Tuning**，
直观展示各方法在可训练参数量与模型效果之间的权衡。

In [17]:
# # 挂载 Google 云盘：将 Google Drive 映射为 Colab 虚拟机上的本地目录
# # 挂载后可像操作本地文件一样读写 Drive 中的文件（sst2 数据集、模型权重等）
# # 运行后会弹出授权链接，点击链接并完成授权即可完成挂载
# from google.colab import drive  # 导入 Colab 专用的 drive 模块，仅在 Google Colab 环境中可用
# drive.mount('/content/drive')   # 将 Google Drive 挂载到 /content/drive 路径下

In [18]:
# # 列出 Google Drive 根目录（MyDrive）下的所有文件和文件夹
# # 确认 sst2 数据集文件夹是否已存在于云盘中
# !ls /content/drive/MyDrive/  # !ls 在 Linux/Colab 中输出目录内容

In [19]:
# # 切换当前工作目录到 Google Drive 的根目录
# # %cd 是 IPython 魔法命令，对整个 Notebook 会话生效，后续所有相对路径均以此为基准
# # 切换后，./sst2/ 等相对路径即可访问 Drive 中的数据集目录
# %cd /content/drive/MyDrive/

In [20]:
# # 列出 sst2 目录内容，确认数据文件（parquet 格式）已正确存放
# # SST-2 是 Stanford Sentiment Treebank 的二分类子集，包含正面/负面情感标签
# # 预期看到 data/train-00000-of-00001.parquet 和 data/validation-00000-of-00001.parquet
# !ls sst2

In [21]:
# # 安装指定版本的 HuggingFace datasets 库（2.20.0），用于加载 parquet 格式的 SST-2 数据集
# # 固定版本是为了保证 API 兼容性，避免新版本引入不兼容变更
# !pip install datasets==2.20.0

In [22]:
%%bash
# 检查当前环境中已安装的 datasets 版本，确认安装成功且版本符合要求
pip list|grep datasets

datasets                  5.0.0


In [23]:
from datasets import load_dataset                                          # HuggingFace datasets：用于加载本地 parquet 数据集文件
from transformers import (                                                  # HuggingFace Transformers 工具
    AutoTokenizer,                                                          #   自动分词器：根据模型名称加载对应分词器
    AutoConfig,                                                             #   自动配置：加载模型超参数配置（如隐藏层大小、层数等）
    AutoModel,                                                              #   自动模型：根据配置加载预训练模型骨干
    get_cosine_schedule_with_warmup                                         #   余弦退火学习率调度器（含 warmup 阶段）
)

import torch                                                                # PyTorch 深度学习框架
import torch.nn as nn                                                       # 神经网络模块（Linear、Module 等）
import torch.nn.functional as F                                             # 函数式接口（binary_cross_entropy 等）
from torch.utils.data import DataLoader                                     # 数据批次加载器，自动打乱、分批、并行读取
import numpy as np                                                          # NumPy 数值计算（参数统计等辅助计算）
from tqdm import tqdm                                                      # 自适应进度条（Jupyter/终端环境均可用）

# 固定随机种子，保证每次运行结果可复现
# torch.manual_seed 设置 CPU 随机数生成器的种子（int），需在创建任何随机张量前调用
torch.manual_seed(42)

# 自动选择计算设备：优先使用 GPU（CUDA），否则回退到 CPU
# device 类型: torch.device，后续所有张量/模型均需移至此设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 最大训练轮数（int）；GPU 充足时可适当增大（如 20）以获得更好效果
num_epochs = 15

# 早停耐心值（int）：验证集准确率连续 patience 轮不提升则停止训练，防止过拟合
patience = 9999

# 训练记录字典：键为方法名（str），值为该方法的 history 字典（含 train_loss/acc、val_loss/acc）
# 所有 PEFT 方法训练完成后统一用此字典绘制对比图
training_record = {}  # 类型: dict[str, dict[str, list[float]]]

## 一、准备工作

SST-2（Stanford Sentiment Treebank）是斯坦福大学发布的一个用于情感分析的数据集，旨在对句子的情感进行分类。该数据集中包含电影评论句子，每个句子都带有相应的情感标签。情感标签分为两类：正面（positive）和负面（negative），因此 SST-2 是一个二分类任务的数据集。

Positive（正面）：

"The movie was absolutely fantastic."（这部电影绝对太棒了。）
"I loved the acting and the storyline."（我喜欢演技和故事情节。）

Negative（负面）：

"The film was a complete disaster."（这部电影完全是个灾难。）
"The acting was terrible, I wouldn't recommend it."（演技糟糕，我不推荐这部电影。）

### 1.1 加载SST-2数据集与构建数据加载器

In [24]:
from pathlib import Path                                                    # pathlib.Path：面向对象的路径操作类，支持 / 运算符拼接子路径
# 参考文档：https://huggingface.co/google-bert/bert-base-uncased

# pathlib.Path：本地缓存根目录对象，首次下载后模型/词表文件保存于此，后续直接从本地加载
# 若该目录不存在，from_pretrained 内部会自动递归创建（os.makedirs exist_ok=True），无需手动 mkdir
# 使用 Path 而非 str 是因为后续通过 / 运算符（Path.__truediv__）拼接子目录路径
CACHE_DIR_DATA = Path("data")                                                     # 类型: pathlib.Path，指向 ./data/ 目录
CACHE_DIR_MODEL=Path("model")

# AutoTokenizer.from_pretrained 接口详解：
# 该类方法会自动根据模型名称推断并实例化对应的分词器类（本例为 BertTokenizerFast）
#
# 参数说明：
#   pretrained_model_name_or_path (str | PathLike, 必填，第一个位置参数)：
#       指定要加载的预训练分词器。支持以下三种形式：
#       ① HuggingFace Hub 模型 ID（如 "bert-base-uncased"）
#          → 从 Hub 下载对应的 tokenizer.json / vocab.txt 等词表文件
#          → 实际下载地址：https://huggingface.co/bert-base-uncased/resolve/main/tokenizer.json
#       ② 本地目录路径（如 "./my_model"）
#          → 从该目录读取 tokenizer_config.json 等配置文件，完全离线运行
#       ③ 单个文件路径（如 "./vocab.txt"）
#          → 直接加载指定词表文件
#       「uncased」含义：模型在预训练时将所有输入转为小写，因此分词器会在 tokenize 前
#       自动调用 .lower()，大小写不敏感（"Hello" 和 "hello" 映射到相同 token）
#
#   cache_dir (str | PathLike, 可选, 默认 None)：
#       本地缓存根目录。行为逻辑：
#       - 若为 None：使用系统默认缓存路径（~/.cache/huggingface/hub/）
#       - 若指定路径且文件已存在：直接从本地读取，跳过网络请求（离线可用）
#       - 若指定路径但文件不存在：先从 Hub 下载，再保存至此路径
#       本例传入 CACHE_DIR_DATA/"sst-2_tokenizer" 即 Path("data")/"sst-2_tokenizer"
#       实际路径为 ./data/sst-2_tokenizer/，分词器词表等文件保存于此
#
#   use_fast (bool, 可选, 默认 True)：
#       是否使用基于 Rust 实现的 Fast 版分词器（BertTokenizerFast）
#       - True（默认）：使用 Rust 加速版，批量编码速度比 Python 版快 10~100 倍
#       - False：使用纯 Python 实现的 BertTokenizer，速度较慢但兼容性更好
#
#   revision (str, 可选, 默认 "main")：
#       指定从 Hub 加载的 Git 分支、标签或 commit SHA，用于版本锁定
#       示例：revision="v4.30.0" 或 revision="abc1234"（commit hash）
#
#   token (str | bool, 可选, 默认 None)：
#       HuggingFace API 访问令牌（Token），用于访问私有或受限模型
#       - None / False：匿名访问（仅公开模型）
#       - True：使用本地  保存的令牌
#       - str：直接传入令牌字符串（如 "hf_xxxx"）
#
#   trust_remote_code (bool, 可选, 默认 False)：
#       是否允许执行 Hub 上模型仓库中的自定义 Python 代码（如自定义分词逻辑）
#       标准模型（bert、gpt2 等）无需开启；仅对含自定义 tokenizer 代码的新模型需要设为 True
#
#   local_files_only (bool, 可选, 默认 False)：
#       是否强制只使用本地缓存，完全禁止网络请求
#       - False（默认）：优先使用缓存，缓存不存在时联网下载
#       - True：仅使用本地缓存，缓存不存在时抛出 OSError，适合离线部署场景
#
# 返回值：
#   tokenizer (transformers.BertTokenizerFast)：
#       BERT 专用的快速分词器实例，核心方法：
#       - tokenizer(texts, ...)           → BatchEncoding（含 input_ids、attention_mask 等）
#       - tokenizer.encode(text)          → list[int]，单句 token ID 列表
#       - tokenizer.decode(ids)           → str，将 token ID 还原为文本
#       - tokenizer.vocab_size            → int，词表大小（bert-base-uncased 为 30522）
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", cache_dir=CACHE_DIR_DATA/"sst-2_tokenizer")

# load_dataset 是通用数据加载接口，第一个参数 path 决定数据来源：
#   ① path = Hub 数据集名（如 "glue"、"imdb"）
#       → 联网从 HuggingFace Hub 下载
#   ② path = 格式名（如 "parquet"、"csv"、"json"）
#       → 读取本地文件，触发内置 Packaged Builder 解析，无需联网
#       → 文件位置通过 data_dir 或 data_files 二选一指定：
#           data_dir  : 传入目录路径，自动扫描目录下所有同格式文件
#                       按文件名推断 split（含 "train"→train，含 "val"/"dev"→validation，含 "test"→test）
#                       自动合并同 split 的多文件（如 train-00000-of-00002 + train-00001-of-00002 → "train"）
#                       示例：load_dataset("parquet", data_dir="./my_data")
#           data_files: 手动显式指定每个 split 的文件路径，适合文件命名不规范或需精确控制的场景
#                       支持列表或通配符，split 名由 dict 的 key 决定（不依赖文件名）
#                       示例：load_dataset("parquet", data_files={"train": "./my_data/train-*.parquet"})
#           注：若同时传入两者，data_files 优先，data_dir 被忽略
#   ③ path = 本地目录路径（如 "./my_data"）
#       → 加载由 dataset.save_to_disk() 保存的数据集，与 ② 完全不同
#
# 本处使用①
dataset_sst2 = load_dataset(
    "stanfordnlp/sst2",       # str：HuggingFace Hub 数据集名称，格式为 "<组织名>/<数据集名>"
                              #      首次运行时自动联网下载，后续从 cache_dir 本地缓存读取
    cache_dir=CACHE_DIR_DATA/"sst-2", # str：本地缓存根目录；下载的文件保存于此，再次加载时直接命中缓存、无需重复下载
    # 返回值 dataset_sst2 类型: datasets.DatasetDict
    #   dataset_sst2["train"]      → datasets.Dataset，约 67349 条样本
    #   dataset_sst2["validation"] → datasets.Dataset，约 872 条样本
    #   dataset_sst2["test"]       → datasets.Dataset，约 1821 条样本
    #
    # 每个 split 的单条样本（dataset[i] / __getitem__ 返回值）类型为 dict，包含以下字段：
    #   {
    #       "sentence": str   — 原始英文电影评论句子，例如 "a stirring , funny and finally transporting re-imagining ..."
    #       "label":    int   — 情感极性标签，0 = 负面（negative），1 = 正面（positive）
    #                           注：test split 的 label 全为 -1（HuggingFace 官方未公开测试集标注）
    #       "idx":      int   — 样本在原始数据集中的全局编号，从 0 开始递增，可用于去重或溯源
    #   }
)


def collate_fn(batch):
    """
    DataLoader 的批次整理函数：将单条样本列表合并为模型可接受的批次张量。

    参数:
        batch (list[dict]): 一个批次的样本列表，每个 dict 包含 "sentence"（str）和 "label"（int）

    返回:
        inputs (transformers.BatchEncoding): 批次化的分词结果，包含：
            - input_ids:      形状 (batch_size, max_seq_len)，token ID 张量
            - attention_mask: 形状 (batch_size, max_seq_len)，1=有效token，0=padding
            - token_type_ids: 形状 (batch_size, max_seq_len)，单句任务全为 0
        labels (torch.Tensor): 形状 (batch_size,)，int64，情感标签（0=负面，1=正面）
    """
    # 对字符串文本批量编码为 token ID
    # padding="longest"  : 将批次内所有序列 padding 到最长序列的长度（动态 padding，减少计算浪费）
    # truncation=True    : 超过 max_length 的序列自动截断
    # return_tensors="pt": 返回 PyTorch 张量格式
    # max_length=512     : BERT 最大支持 512 个 token，超过则截断
    inputs = tokenizer(
        [x["sentence"] for x in batch],    # 提取每个样本的句子文本，类型: list[str]
        padding="longest",                  # 动态 padding 到批次最长序列长度 L（L ≤ 512）
        truncation=True,                    # 超过 max_length 的序列自动截断
        return_tensors="pt",               # 返回 PyTorch 张量格式
        max_length=512                      # BERT 最大序列长度限制
    )
    # inputs 类型: transformers.BatchEncoding，包含以下键（均为 torch.Tensor）：
    #   inputs["input_ids"]      形状: (batch_size, L)，int64，每个 token 对应词表中的 ID
    #   inputs["attention_mask"] 形状: (batch_size, L)，int64，1=真实 token，0=padding 位置
    #   inputs["token_type_ids"] 形状: (batch_size, L)，int64，单句任务全为 0（区分句对中的句A/句B）
    # 其中 L = 批次内最长序列实际长度（padding="longest" 的结果），L ≤ 512
    # 将整数标签列表转换为 int64 张量，形状: (batch_size,)
    labels = torch.tensor([x["label"] for x in batch])
    return inputs, labels  # 返回: (BatchEncoding, Tensor)


# 构建训练集 DataLoader
# batch_size=32    : 每批加载 32 个样本（int）
# shuffle=True     : 每个 epoch 开始前随机打乱训练集，防止模型记住样本顺序
# collate_fn       : 使用自定义整理函数
# train_loader 类型: torch.utils.data.DataLoader，迭代产出 (BatchEncoding, Tensor) 元组
train_loader = DataLoader(dataset_sst2["train"], batch_size=64, shuffle=True, collate_fn=collate_fn)

# 构建验证集 DataLoader（不需要 shuffle，保证每次验证结果一致）
# val_loader 类型: torch.utils.data.DataLoader，迭代产出 (BatchEncoding, Tensor) 元组
val_loader = DataLoader(dataset_sst2["validation"], batch_size=128, collate_fn=collate_fn)


In [25]:
# 查看训练集样本总数
# dataset_sst2["train"] 类型: datasets.Dataset
# len() 返回类型: int，即训练集样本数量（SST-2 训练集约 67349 条）
len(dataset_sst2["train"])

67349

In [26]:
# 从训练集 DataLoader 中取出第一个批次，打印各字段内容以验证数据格式
for inputs, label in train_loader:
    # inputs['input_ids'][0] : 第 0 个样本的 token ID 序列，类型: torch.Tensor，形状: (seq_len,)
    print(inputs['input_ids'][0])

    # inputs['input_ids'][1] : 第 1 个样本的 token ID 序列（与第 0 个对比，观察 padding 效果）
    print(inputs['input_ids'][1])

    # inputs['attention_mask'].shape : 掩码张量的形状 (batch_size, max_seq_len)
    # 1 表示有效 token，0 表示 padding 位置（不参与注意力计算）
    print(inputs['attention_mask'].shape)

    # inputs['token_type_ids'][0] : 第 0 个样本的句子类型 ID，类型: torch.Tensor，形状: (seq_len,)
    # 对于单句分类任务，token_type_ids 全为 0（0=句子A，1=句子B，单句只有句子A）
    print(inputs['token_type_ids'][0])

    # label : 当前批次的标签张量，类型: torch.Tensor，形状: (batch_size,)，值为 0（负面）或 1（正面）
    print(label)

    break  # 只查看第一个批次后立即退出循环

tensor([ 101, 2062, 8669, 1010, 2062, 6832,  102,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0])
tensor([  101,  2009,  1005,  1055, 10889, 11519,  1010,  3391,  2005,  1037,
         7891, 18932,  1999,  1037,  2186,  1012,   102,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0])
torch.Size([64, 35])
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
tensor([1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0,
        1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1])


### 1.2 定义训练与评估函数

In [27]:
def evaluate(model, val_loader):
    """
    在验证集上评估模型的损失和准确率。

    参数:
        model (nn.Module): 待评估的分类模型，前向传播需返回形状 (batch_size,) 的概率值
        val_loader (DataLoader): 验证集数据加载器，每批产出 (BatchEncoding, Tensor) 元组

    返回:
        val_loss (float): 验证集上所有批次的平均二元交叉熵损失
        val_acc  (float): 验证集上的整体准确率（正确预测数 / 总样本数）
    """
    model.eval()                # 切换为评估模式，关闭 Dropout 等训练专用层
    val_loss = 0                # 累积所有批次的损失值之和（float）
    val_acc = 0                 # 累积所有批次中预测正确的样本数（int）
    with torch.no_grad():       # 在评估过程中关闭梯度计算，节省显存、加速推理
        total_samples = 0       # 统计验证集总样本数量（int）
        for inputs, labels in val_loader:
            # 将 BatchEncoding 中所有张量字段移至目标设备（字典推导式）
            # inputs 原为 CPU 张量，.to(device) 移至 GPU
            inputs = {k: v.to(device) for k, v in inputs.items()}
            labels = labels.to(device)           # 标签张量移至目标设备，形状: (batch_size,)

            probs = model(**inputs)              # 前向传播，**inputs 展开 BatchEncoding 字典
            # probs 类型: torch.Tensor，形状: (batch_size,)，值域 [0, 1]（sigmoid 输出概率）

            # ── 三种常用损失函数 input/target shape 对比 ──────────────────────────────
            # F.binary_cross_entropy(input, target)
            #   input  : FloatTensor，shape (N,) 或 (N, *)，须先经过 sigmoid，值域 [0, 1]
            #   target : FloatTensor，shape 与 input 相同，值为 0.0 / 1.0
            #
            # F.binary_cross_entropy_with_logits(input, target)
            #   input  : FloatTensor，shape (N,) 或 (N, *)，原始 logits（无需 sigmoid）
            #   target : FloatTensor，shape 与 input 相同，值为 0.0 / 1.0
            #   优势   : 内部用 log-sum-exp 计算，数值更稳定，推荐替代 binary_cross_entropy
            #
            # F.cross_entropy(input, target)
            #   input  : FloatTensor，shape (N, C)，C 为类别数，原始 logits（无需 softmax）
            #   target : LongTensor， shape (N,)，值为类别索引 [0, C-1]
            #   适用   : 多分类任务
            # ─────────────────────────────────────────────────────────────────────────
            loss = F.binary_cross_entropy(probs, labels.float())
            val_loss += loss.item()              # .item() 将标量张量转为 Python float，累加

            # (probs > 0.5) : bool 张量，形状 (batch_size,)，True=预测为正类
            # == labels     : 与真实标签比较，bool 张量，True=预测正确
            # .sum().item() : 统计当前批次预测正确的样本数（int）
            val_acc += ((probs > 0.5) == labels).sum().item()
            total_samples += len(labels)         # 累积当前批次的样本数

    val_loss /= len(val_loader)  # 除以批次数，得到所有批次的平均损失（float）
    val_acc /= total_samples     # 除以总样本数，得到整体准确率（float，范围 [0, 1]）
    return val_loss, val_acc     # 返回: (平均损失: float, 准确率: float)


def train(model, train_loader, val_loader, device, num_epochs=3, patience=3):
    """
    训练分类模型，支持余弦学习率调度和早停策略。

    参数:
        model        (nn.Module)    : 待训练的分类模型
        train_loader (DataLoader)   : 训练集数据加载器
        val_loader   (DataLoader)   : 验证集数据加载器
        device       (torch.device) : 计算设备（"cuda" 或 "cpu"）
        num_epochs   (int)          : 最大训练轮数，默认 3
        patience     (int)          : 早停耐心值，验证准确率连续 patience 轮不提升则停止，默认 3

    返回:
        history (dict): 训练过程记录字典，包含以下键：
            - "train_loss": list[float]，每轮训练集平均损失
            - "train_acc":  list[float]，每轮训练集准确率
            - "val_loss":   list[float]，每轮验证集平均损失
            - "val_acc":    list[float]，每轮验证集准确率
    """
    model.to(device)    # 将模型所有参数移至目标设备（GPU/CPU）

    # AdamW 优化器：Adam 变体，内置权重衰减（解耦权重衰减与梯度更新）
    # model.parameters() 返回所有 requires_grad=True 的参数迭代器
    # lr=1e-5 : 学习率（float），微调 BERT 时通常使用较小的学习率（1e-5 ～ 3e-5）
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

    # 总训练步数 = 轮数 × 每轮批次数，用于计算学习率调度器的关键节点
    total_steps = num_epochs * len(train_loader)  # 类型: int

    # 余弦退火学习率调度器：学习率先从 0 线性增大（warmup），再余弦衰减至 0
    # num_warmup_steps : warmup 阶段步数（前 20% 步线性提升学习率）
    # num_training_steps: 总步数，调度器据此计算余弦曲线
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.2 * total_steps),  # 前 20% 步执行 warmup（学习率线性上升）
        num_training_steps=total_steps            # 总训练步数
    )

    best_val_acc = -1   # 记录历史最优验证准确率（float），初始设为 -1（任何准确率都高于它）
    cur = 0             # 当前已连续未提升的轮数计数器（int），达到 patience 时触发早停

    # 历史记录字典，每个键对应一个 list，每轮 append 一个值
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    # tqdm 上下文管理器：total = 总步数（epoch 数 × 每 epoch 批次数），每处理一个批次手动 update(1)
    with tqdm(total=num_epochs * len(train_loader), desc="训练进度") as pbar:
        for epoch in range(num_epochs):  # 遍历每个训练轮次（epoch: int，从 0 开始）
            model.train()       # 切换为训练模式，激活 Dropout 等训练专用层
            train_loss = 0      # 当前轮次的累积训练损失（float）
            train_acc = 0       # 当前轮次的累积训练正确预测数（int）
            total_samples = 0   # 当前轮次的累积训练样本数（int）

            for inputs, labels in train_loader:  # 遍历当前 epoch 的每个批次
                inputs = {k: v.to(device) for k, v in inputs.items()}  # BatchEncoding 各字段移至设备
                labels = labels.to(device)          # 标签张量移至目标设备

                optimizer.zero_grad()               # 清空上一步的梯度缓存，防止梯度累积
                probs = model(**inputs)             # 前向传播：**inputs 将 BatchEncoding 解包为关键字参数
                # probs 类型: torch.Tensor，形状: (batch_size,)

                # ── 三种常用损失函数 input/target shape 对比 ──────────────────────────────
                # F.binary_cross_entropy(input, target)
                #   input  : FloatTensor，shape (N,) 或 (N, *)，须先经过 sigmoid，值域 [0, 1]
                #   target : FloatTensor，shape 与 input 相同，值为 0.0 / 1.0
                #
                # F.binary_cross_entropy_with_logits(input, target)
                #   input  : FloatTensor，shape (N,) 或 (N, *)，原始 logits（无需 sigmoid）
                #   target : FloatTensor，shape 与 input 相同，值为 0.0 / 1.0
                #   优势   : 内部用 log-sum-exp 计算，数值更稳定，推荐替代 binary_cross_entropy
                #
                # F.cross_entropy(input, target)
                #   input  : FloatTensor，shape (N, C)，C 为类别数，原始 logits（无需 softmax）
                #   target : LongTensor， shape (N,)，值为类别索引 [0, C-1]
                #   适用   : 多分类任务
                # ─────────────────────────────────────────────────────────────────────────
                loss = F.binary_cross_entropy(probs, labels.float())
                loss.backward()                     # 反向传播：计算所有可训练参数的梯度
                optimizer.step()                    # 根据梯度更新可训练参数（AdamW 更新规则）
                scheduler.step()                    # 更新学习率（按余弦调度曲线）

                train_loss += loss.item()           # 累积批次损失
                train_acc += ((probs > 0.5) == labels).sum().item()  # 累积正确预测数
                total_samples += len(labels)        # 累积批次样本数

                pbar.update(1)  # 当前批次处理完毕，进度条前进一步

            train_loss /= len(train_loader)         # 当前轮次训练集平均损失（float）
            train_acc  /= total_samples             # 当前轮次训练集准确率（float）

            val_loss, val_acc = evaluate(model, val_loader)  # 在验证集上评估，返回 (float, float)

            # 更新进度条后缀，展示本轮训练和验证指标
            pbar.set_postfix(
                epoch=epoch,                          # 当前轮次（int）
                train_loss=f"{train_loss:.4f}",       # 当前轮训练损失（float → 字符串）
                train_acc=f"{train_acc:.4f}",         # 当前轮训练准确率（float → 字符串）
                val_loss=f"{val_loss:.4f}",           # 当前轮验证损失（float → 字符串）
                val_acc=f"{val_acc:.4f}"              # 当前轮验证准确率（float → 字符串）
            )


            # 将当前轮次的各指标追加到历史记录列表
            history["train_loss"].append(train_loss)
            history["train_acc"].append(train_acc)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)

            # 早停逻辑：若验证准确率创历史新高则重置计数器，否则计数器 +1
            if val_acc > best_val_acc:
                best_val_acc = val_acc  # 更新最优验证准确率
                cur = 0                 # 重置连续未提升计数器
            else:
                cur += 1                # 准确率未提升，计数器加 1
            if cur >= patience:         # 达到耐心值，触发早停
                print("提前停止训练")
                break

    return history  # 返回: dict，包含所有已完成轮次的训练/验证损失和准确率列表

### 1.3 定义参数统计辅助函数

In [28]:
def human_readable_count(n):
    """
    将整数参数量格式化为人类可读的带单位字符串（K / M / B）。

    参数:
        n (int): 参数数量

    返回:
        str: 格式化后的字符串，如 "109.48M"、"6.70B"
    """
    # 注：数字中的下划线 _ 是 Python 3.6+ 的数字分隔符（PEP 515），仅用于视觉分组，不影响数值
    #     1_000 == 1000，1_000_000 == 1000000，写法类似其他语言的千位逗号
    if n < 1_000:                            # 1_000 即 1000
        return f"{n}"                        # 小于 1000 直接显示原始数字（str）
    elif n < 1_000_000:                      # 1_000_000 即 1000000
        return f"{n/1_000:.2f}K"             # 千级别，以 K 为单位（如 "123.45K"）
    elif n < 1_000_000_000:                  # 1_000_000_000 即 1000000000
        return f"{n/1_000_000:.2f}M"         # 百万级别，以 M 为单位（如 "109.48M"）
    else:
        return f"{n/1_000_000_000:.2f}B"     # 十亿级别，以 B 为单位（如 "6.70B"）


def count_parameters(model):
    """
    统计并打印模型的总参数量、冻结参数量和可训练参数量。

    参数:
        model (nn.Module): 需要统计参数的 PyTorch 模型

    返回:
        None（直接打印结果到标准输出）
    """
    # p.numel() 返回参数张量中元素（标量值）的总个数（int）
    # 对所有参数张量的元素数求和，得到模型总参数量（int）
    total_params = sum(p.numel() for p in model.parameters())

    # 只统计 requires_grad=True 的参数（即在反向传播中会被更新的参数）
    # 可训练参数量 (int)：对应 PEFT 中实际参与训练的参数
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # 冻结参数量（int） = 总参数量 - 可训练参数量
    frozen_params = total_params - trainable_params

    # :>8 格式化：右对齐，字段宽度 8，使输出列对齐
    print(f"Total Parameters:\t{human_readable_count(total_params):>8}")       # 总参数量
    print(f"Frozen Parameters:\t{human_readable_count(frozen_params):>8}")     # 冻结参数量
    # :.2% 格式化为百分比（如 0.0041 → "0.41%"），直观展示 PEFT 方法的参数效率
    print(f"Trainable Parameters:\t{human_readable_count(trainable_params):>8}\t{trainable_params / total_params:.2%}")

In [29]:
import matplotlib.pyplot as plt  # 导入绘图库

def plot_training_record(training_record, metric_name="val_acc"):
    """
    绘制训练记录图表

    参数:
    training_record(dict): 包含训练记录的字典，键为方法名称，值为记录的字典
    metric_name(str): 要绘制的度量名称，默认为"val_acc"（验证准确度）

    返回:
    无（直接展示图表）
    """
    plt.figure(figsize=(12, 6))  # 设置图表大小
    for method_name, record in training_record.items():  # 遍历每个方法的记录
        metrics = record[metric_name]  # 获取指定度量的数值
        plt.plot(range(len(metrics)), metrics, label=method_name)  # 绘制折线图

    plt.xlabel("Epoch")  # 设置X轴标签
    plt.ylabel("Validation Accuracy")  # 设置Y轴标签
    plt.legend()  # 显示图例
    plt.grid()  # 显示网格线
    plt.show()  # 展示图表

In [30]:
# 加载 BERT-base-uncased 预训练模型骨干（不含分类头）
# ── 模型命名规则 ──
# uncased : 不区分大小写；分词前统一将输入转为小写
#           例如 'Hello' 与 'hello' 映射到相同词元，适合对大小写不敏感的通用 NLP 任务
# cased   : 保留原始大小写；'Hello' 与 'hello' 是不同词元，适合命名实体识别等大小写敏感任务
#
# ── AutoModel.from_pretrained 接口说明 ──
# AutoModel 会读取模型目录下 config.json 中的 model_type 字段，自动推断并实例化对应模型类
#   model_type='bert'  → BertModel
#   model_type='gpt2'  → GPT2Model
# 因此换模型时只需修改名称，其余代码无需改动
#
# 主要参数说明:
#   pretrained_model_name_or_path : str/Path  Hub 模型 ID 或本地目录路径
#   cache_dir                     : str/Path  权重缓存目录，命中后不重复下载；
#                                             不指定则默认存至 ~/.cache/huggingface/hub
#   torch_dtype                   : torch.dtype  加载精度，float16/bfloat16 可降低显存，默认 float32
#   device_map                    : str/dict  设备映射，'auto' 自动分配层到可用 GPU/CPU
#   ignore_mismatched_sizes       : bool  形状不匹配时跳过而非报错，默认 False
#   trust_remote_code             : bool  加载含自定义架构的模型时需设为 True（注意安全风险）
#   local_files_only              : bool  仅使用本地缓存，不联网，默认 False
#   token                         : str   HuggingFace Hub 访问令牌，加载私有模型时需要
#   revision                      : str   模型版本，可指定 commit hash 或分支名，默认 'main'
#
# 返回值: transformers.BertModel（继承自 nn.Module）
#   - 默认以 float32 加载，约占 440 MB 内存
#   - 默认处于 eval() 模式；需要训练时须手动调用 .train()
#   - 仅含 BERT 编码器主干，不含 MLM/NSP 预训练任务头
#     （加载时 cls.predictions.* / cls.seq_relationship.* 会报 UNEXPECTED，属正常现象，可忽略）
bert_model = AutoModel.from_pretrained("bert-base-uncased", cache_dir=CACHE_DIR_MODEL/"bert-base-uncased")

# 打印模型完整结构，包括嵌入层、12 个 BertLayer、及最终 BertPooler
print(bert_model)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropou

## 二、冻结BERT作为特征提取器（Frozen BERT）

**思路**：完全冻结预训练 BERT 的所有参数，仅在其上层添加一个可训练的线性分类头。BERT 的输出仅作为固定特征（feature extraction），不随下游任务更新，可训练参数极少。

In [31]:
# 参考文档：https://huggingface.co/docs/transformers/model_doc/bert
class FrozenBert(nn.Module):
    """
    冻结 BERT 特征提取器：固定 BERT 全部参数，仅训练顶层线性分类头。

    可训练参数量 ≈ hidden_size × 1 + 1 = 768 × 1 + 1 = 769（bias=True 默认）。
    适合快速验证预训练特征的线性可分性，但通常效果低于微调方法。

    参数:
        无（所有超参数在 __init__ 内部硬编码或从模型配置中读取）
    """
    def __init__(self):
        super().__init__()
                # 加载预训练 BERT-base-uncased 模型（下载约 440 MB，需等待）
        # cache_dir指定本地缓存目录，首次下载后直接命中缓存
        # self.model 类型: transformers.BertModel，输出隐藏层特征
        self.model = AutoModel.from_pretrained("bert-base-uncased", cache_dir=CACHE_DIR_MODEL/"bert-base-uncased")

        # 添加线性分类头：将 [CLS] token 的 hidden_size 维特征映射为 1 个输出值
        # hidden_size = 768（bert-base 的隐藏层维度）
        # 输出维度为 1，配合 sigmoid 输出二分类概率
        # self.classifier 类型: nn.Linear，输入维度 768，输出维度 1
        self.classifier = nn.Linear(self.model.config.hidden_size, 1)

        # 冻结 BERT 所有参数：将 requires_grad 设为 False，反向传播时不计算梯度也不更新权重
        # 冻结后这些参数不消耗梯度内存，显著降低显存占用
        for param in self.model.parameters():
            param.requires_grad = False  # False = 冻结（不参与训练）

    def forward(self, **inputs):
        """
        前向传播：提取 [CLS] token 特征并输出分类概率。

        参数:
            **inputs: BatchEncoding 展开的关键字参数，包括 input_ids、attention_mask、token_type_ids

        返回:
            torch.Tensor: 形状 (batch_size,)，值域 [0,1]，表示正类（正面情感）的概率
        """
        # BERT 前向传播，返回 BaseModelOutputWithPoolingAndCrossAttentions 对象
        # .last_hidden_state : 最后一层所有 token 的隐藏状态，形状: (batch_size, seq_len, hidden_size)
        # [:, 0, :]          : 取第 0 个 token（即 [CLS] token）的表示，形状: (batch_size, hidden_size)
        # [CLS] token 的输出被设计为聚合整句语义，适合用于分类任务
        feature = self.model(**inputs).last_hidden_state[:, 0, :]

        # 线性层将 hidden_size 维特征映射为标量 logit，形状: (batch_size, 1)
        logits = self.classifier(feature)

        # sigmoid 将 logit 转换为 [0,1] 概率，.squeeze() 去掉最后一个维度 1
        # 返回形状: (batch_size,)
        return torch.sigmoid(logits).squeeze()


frozen_bert = FrozenBert()     # 实例化模型，加载预训练权重并冻结 BERT 参数
print(frozen_bert)             # 打印模型层级结构，确认分类头已挂载
count_parameters(frozen_bert)  # 打印总参数量、冻结参数量和可训练参数量

# training_record 已在全局初始化为 {}，此处无需重新定义，直接用键名写入结果即可



FrozenBert(
  (model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [32]:
%%time

# 训练冻结 BERT 模型，只有分类头（769 个参数）参与梯度更新
# 训练完成后将 history 字典写入 training_record 以键 "Frozen" 标识
# training_record["Frozen"] 类型: dict[str, list[float]]，含 train_loss/acc、val_loss/acc
training_record["Frozen"] = train(
    frozen_bert,       # 待训练模型（仅分类头可训练）
    train_loader,      # 训练集 DataLoader
    val_loader,        # 验证集 DataLoader
    device,            # 计算设备（GPU/CPU）
    num_epochs=num_epochs,  # 最大轮数（全局变量，默认 5）
    patience=patience       # 早停耐心值（全局变量，默认 5）
)

训练进度:  40%|███▉      | 6305/15795 [04:58<07:29, 21.10it/s, epoch=4, train_acc=0.7099, train_loss=0.6111, val_acc=0.7626, val_loss=0.5906]


CPU times: total: 6min 50s
Wall time: 4min 59s


KeyboardInterrupt: 

In [ ]:
# 直接输出 training_record 字典，查看 "Frozen" 方法每轮的训练/验证指标数值
# 类型: dict[str, dict[str, list[float]]]
# Jupyter 会自动格式化字典并在输出区渲染
training_record
del frozen_bert  # 删除已训练完毕的 FrozenBert 模型，释放显存

In [ ]:
# 绘制当前已完成训练的所有方法的验证准确率曲线（折线图）
# metric_name="val_acc" : 指定绘制验证集准确率（val_acc）随 epoch 的变化趋势
# 每条线对应一种 PEFT 方法，颜色和图例由 plot_training_record 自动处理
plot_training_record(training_record, metric_name="val_acc")

## 三、全量微调（Full Fine-Tuning）

> ⚠️ **本节不是重点，了解即可，可跳过。**
> 全量微调不属于参数高效微调（PEFT）的范畴，与本 Notebook 的核心主题关联较弱。
> 相比之下，后续的 BitFit、P-Tuning、Prefix Tuning、LoRA、Adapter 等方法才是 PEFT 的核心，建议重点学习。

**思路**：解冻 BERT 所有参数，所有层均参与梯度更新。效果最好，但可训练参数量与总参数量相同（约 110M），显存消耗大，且容易遗忘预训练知识（灾难性遗忘）。

In [ ]:
%%time

class FullyFinetunedBert(nn.Module):
    """
    全量微调 BERT：BERT 所有层的参数全部参与训练更新。

    与 FrozenBert 相比，去掉了参数冻结步骤，所有参数默认 requires_grad=True。
    可训练参数量 ≈ 109.48M（BERT-base 全部参数） + 769（分类头）。
    通常效果最优，但显存消耗大（约 2× 冻结版本）。
    """
    def __init__(self):
        super().__init__()
        # 加载预训练 BERT-base-uncased，所有参数默认 requires_grad=True（全量微调）
        self.model = AutoModel.from_pretrained("bert-base-uncased", cache_dir=CACHE_DIR_MODEL/"bert-base-uncased")

        # 线性分类头：将 [CLS] token 的 768 维特征映射为 1 个输出值
        self.classifier = nn.Linear(self.model.config.hidden_size, 1)

    def forward(self, **inputs):
        """
        前向传播：提取 [CLS] 特征并输出分类概率。

        参数:
            **inputs: BatchEncoding 展开的关键字参数

        返回:
            torch.Tensor: 形状 (batch_size,)，值域 [0,1]，正类概率
        """
        # 取最后一层第 0 个 token（[CLS]）的隐藏状态作为句子级特征
        # feature 形状: (batch_size, hidden_size=768)
        feature = self.model(**inputs).last_hidden_state[:, 0, :]
        logits = self.classifier(feature)        # 形状: (batch_size, 1)
        return torch.sigmoid(logits).squeeze()   # 形状: (batch_size,)，值域 [0,1]


fully_fine_tuned_bert = FullyFinetunedBert()  # 实例化全量微调模型
print('-' * 50)
print(fully_fine_tuned_bert)                  # 打印模型结构
print('-' * 50)
count_parameters(fully_fine_tuned_bert)       # 打印参数统计（可训练参数 ≈ 110M）

# 训练全量微调模型，所有层参数均参与更新，训练结果记录到 training_record
training_record["Fully Fine-Tuning"] = train(
    fully_fine_tuned_bert,   # 全量微调模型
    train_loader,            # 训练集 DataLoader
    val_loader,              # 验证集 DataLoader
    device,                  # 计算设备
    num_epochs=num_epochs,   # 最大训练轮数
    patience=patience        # 早停耐心值
)

In [ ]:
# 删除全量微调模型对象，释放 GPU 显存（约 440 MB），为后续方法腾出空间
# del 语句解除变量引用，Python 垃圾回收机制自动释放内存/显存
del fully_fine_tuned_bert

In [ ]:
# 绘制 Frozen 和 Fully Fine-Tuning 两种方法的验证准确率对比曲线
# 可以看到全量微调的效果（val_acc）通常显著优于冻结特征提取方法
plot_training_record(training_record, metric_name="val_acc")

## 四、BitFit：仅微调偏置项

**论文**：BitFit: Simple Parameter-efficient Fine-tuning for Transformer-based Masked Language-models

**核心思想**：只更新模型中的 **bias（偏置项）** 参数，冻结所有权重矩阵（weight）。偏置项参数量仅占 BERT-base 总参数的约 **0.08%**，但在许多 NLU 任务上可以达到接近全量微调的效果。

In [ ]:
%%time

class BitFitBert(nn.Module):
    """
    BitFit 方法：冻结所有权重矩阵（weight），仅训练偏置项（bias）。

    实现方式：遍历所有命名参数，凡参数名中不含 "bias" 字符串的参数均冻结。
    可训练参数量 ≈ BERT-base 所有 bias 参数之和（约 90K）+ 分类头参数（769）
                 ≈ 总参数量的 0.08%。
    """
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained("bert-base-uncased", cache_dir=CACHE_DIR_MODEL/"bert-base-uncased")  # 加载预训练 BERT
        # 线性分类头：768 维 → 1 维输出（分类概率）
        self.classifier = nn.Linear(self.model.config.hidden_size, 1)

        # 遍历所有命名参数，冻结非偏置项参数（权重矩阵等）
        # name : str，参数名（如 "encoder.layer.0.attention.self.query.weight"）
        # param: nn.Parameter，参数张量
        # 名称中包含 "bias" 的参数（偏置项）保持 requires_grad=True，可参与更新
        # 名称中不含 "bias" 的参数（权重矩阵）设为 requires_grad=False，冻结
        for name, param in self.model.named_parameters():
            if "bias" not in name:
                param.requires_grad = False  # 冻结权重矩阵参数

    def forward(self, **inputs):
        """
        前向传播：取 [CLS] 特征，经分类头输出正类概率。

        参数:
            **inputs: BatchEncoding 展开的关键字参数

        返回:
            torch.Tensor: 形状 (batch_size,)，值域 [0,1]
        """
        # 取最后一层 [CLS] token 隐藏状态，形状: (batch_size, hidden_size)
        feature = self.model(**inputs).last_hidden_state[:, 0, :]
        logits = self.classifier(feature)         # 形状: (batch_size, 1)
        return torch.sigmoid(logits).squeeze()    # 形状: (batch_size,)


bitfit_bert = BitFitBert()       # 实例化 BitFit 模型（仅偏置项可训练）
print('-' * 50)
print(bitfit_bert)               # 打印模型结构（与全量微调相同，差异在参数的 requires_grad 状态）
print('-' * 50)
count_parameters(bitfit_bert)    # 打印参数统计，可验证可训练参数极少（约 90K）

# 训练 BitFit 模型并记录结果
training_record["BitFit"] = train(
    bitfit_bert, train_loader, val_loader, device,
    num_epochs=num_epochs, patience=patience
)

In [ ]:
# 释放 BitFit 模型占用的显存，为后续 P-Tuning 等方法腾出空间
del bitfit_bert

In [ ]:
# 对比 Frozen / Full Fine-Tuning / BitFit 三种方法的验证准确率曲线
# 偏置项微调（BitFit）效果通常介于冻结特征提取和全量微调之间，但可训练参数极少
plot_training_record(training_record, metric_name="val_acc")

## 五、提示调优系列（P-Tuning / Prefix Tuning / P-Tuning v2）

> **核心设计思想**：用可学习的「软提示」（soft prompt）替代人工设计的「硬提示」（hard prompt），通过梯度下降自动找到最优的任务引导向量，同时冻结预训练语言模型（PLM）全部权重。三者的演进只在于软提示注入的位置越来越深：P-Tuning 仅在输入嵌入层插入；Prefix Tuning 和 P-Tuning v2 在每层 K/V 均注入，影响更深、效果更强。
>
> - **硬提示**：人工用自然语言编写的离散 token，不可微，只能靠人工经验调整，效果依赖措辞。
> - **软提示**：本质是插入输入序列中的**虚拟 token**——没有对应的真实词汇，只是嵌入空间中的连续向量，与真实 token 的嵌入向量形状相同、位置相邻，因此可无缝拼入输入。其优势在于可被梯度直接优化，自动在嵌入空间中搜索对当前任务最有引导效果的方向。
> - **为何有效**：预训练模型已将各种任务能力压缩编码在权重中，但它不知道当前要激活哪种能力。虚拟 token（任务引导向量）通过注意力机制影响每个真实 token 的表示，引导模型在前向计算时走向目标任务对应的特征方向，从而将潜伏的任务能力显式激活——模型能力本已具备，虚拟 token 是将其唤醒的任务信号。

Prefix Tuning（论文：Prefix-Tuning: Optimizing Continuous Prompts for Generation），在输入token之前构造一段任务相关的virtual tokens作为Prefix，然后训练的时候只更新Prefix部分的参数，而PLM中的其他部分参数固定。

针对不同的模型结构，需要构造不同的Prefix。
* 针对自回归架构模型：在句子前面添加前缀，得到 z = [PREFIX; x; y]，合适的上文能够在固定 LM 的情况下去引导生成下文（比如：GPT3的上下文学习）。
* 针对编码器-解码器架构模型：Encoder和Decoder都增加了前缀，得到 z = [PREFIX; x; PREFIX'; y]（PREFIX 和 PREFIX' 是两组独立的前缀参数）。Encoder端增加前缀是为了引导输入部分的编码，Decoder 端增加前缀是为了引导后续token的生成。


P-Tuning（论文：GPT Understands, Too），该方法将 Prompt 转换为可学习的虚拟 token（Embedding 层中的连续向量），并提供两种重参数化方式对虚拟 token 进行编码：① **LSTM + MLP**：先用双向 LSTM 对虚拟 token 序列建模，再通过 MLP 映射；② **仅 MLP**：直接用 MLP 对虚拟 token 做非线性变换。重参数化的目的是防止直接优化裸向量时收敛困难。

相比 Prefix Tuning，P-Tuning 的虚拟 token 同样是**拼接在真实 token 之前**（作为前缀），但仅作用于输入嵌入层，不在每一层都注入；此外，原论文支持将虚拟 token 插入模板的任意位置（不局限于句首），灵活性更高。这里的出发点是把传统人工设计模板中的真实 token 替换成可微的虚拟 token，让模型自动学习最优的提示形式。

P-Tuning v2（论文： P-Tuning v2: Prompt Tuning Can Be Comparable to Fine-tuning Universally Across Scales and Tasks），该方法在每一层都加入了Prompts tokens作为输入，而不是仅仅加在输入层，这带来两个方面的好处：

* 更多可学习的参数（从P-tuning和Prompt Tuning的0.01%增加到0.1%-3%），同时也足够参数高效。
* 加入到更深层结构中的Prompt能给模型预测带来更直接的影响。

> **为什么 P-Tuning v2 向 K/V 注入虚拟 token，而不是 Q？**
>
> 在 Self-Attention 中，Q 决定"当前 token 想关注什么"，修改 Q 会直接干扰真实 token 自身的查询语义，破坏程度大。
> 而向 K/V 中追加虚拟 token，相当于**扩大了每个 token 注意力可参考的信息范围**，真实 token 的 Q 保持不变，只是"看的范围"变宽，干扰更小、更可控。
> 此外，向 Q 中注入会导致**残差连接尺寸不一致**：注意力输出序列长度变为 `seq_len + num_virtual`，而残差输入 `x` 的序列长度仍为 `seq_len`，两者无法相加；注入 K/V 则无此问题，因为注意力输出的序列长度由 Q 决定，仍保持 `seq_len`，残差连接正常。
> 最后，HuggingFace 提供了原生 `past_key_values` 接口（原为 KV-Cache 设计），P-Tuning v2 直接复用，**无需侵入修改模型内部结构**，工程实现更简洁。

三种方法对比

> NLG：自然语言生成；NLU：自然语言理解

|方法|参数重整化| 微调参数所在层 | 适配下游任务 |
|-|-|-|-|
|P-Tuning|LSTM + MLP 或 MLP| 仅 embedding 层| NLU（使 GPT 等自回归模型具备 NLU 能力）|
|P-Tuning v2|不使用| 每一层 | NLG & NLU |
|Prefix Tuning|MLP| 每一层 | NLG |

### 5.1 P-Tuning（输入嵌入层软提示）

**思路**：仅在输入层插入若干可学习的「虚拟 token」（Prompt Token），用 LSTM/MLP 对这些虚拟 token 的嵌入进行重参数化，通过梯度下降自动学习最优提示向量，同时冻结 BERT 所有权重。

In [ ]:
import torch        # 导入 PyTorch
import torch.nn as nn  # 导入神经网络模块

# 演示前缀 token 的形状变换：单个前缀参数如何扩展到批次维度
# nn.Parameter 将张量包装为可学习参数（requires_grad=True）
# torch.zeros(20, 768) : 初始化 20 个虚拟 token，每个 token 的嵌入维度为 768
# prefix_tokens 类型: nn.Parameter，形状: (20, 768)
prefix_tokens = nn.Parameter(torch.zeros(20, 768))

# .unsqueeze(0) : 在第 0 维插入一个新维度，形状变为 (1, 20, 768)
# .expand(32, -1, -1) : 沿第 0 维扩展到批次大小 32，-1 表示该维度保持不变
# 注意：expand 不会分配新内存，结果为共享底层存储的视图（view）
# prefix_tokens 形状变为: (32, 20, 768)，即批次中每个样本都附带一组 20 个前缀 token
prefix_tokens = prefix_tokens.unsqueeze(0).expand(32, -1, -1)

# 输出: torch.Size([32, 20, 768])，验证批次扩展结果
print(prefix_tokens.shape)

In [ ]:
import torch                          # 导入 PyTorch 核心库，提供张量运算与自动微分
import torch.nn as nn                  # 导入神经网络模块，包含 Linear、LSTM、Sequential 等层
from transformers import AutoModel     # 导入 HuggingFace AutoModel，用于自动加载预训练模型
import torch.nn.functional as F        # 导入函数式接口（激活函数、损失函数等），此处备用
import numpy as np                     # 导入 NumPy，用于数值计算，此处备用

class PTuningBert(nn.Module):
    """
    P-Tuning 模型：基于 BERT 的软提示微调（Soft Prompt Tuning）分类器。

    在输入嵌入层前插入 num_virtual_tokens 个可学习的虚拟 token（soft prompt），
    BERT 主体参数完全冻结，只训练虚拟 token 及重参数化头（MLP 或 LSTM+MLP）。

    Args:
        num_virtual_tokens (int): 虚拟提示 token 的数量，默认 20。
        reparameterization_type (str): 重参数化方式，"MLP" 或 "LSTM"，默认 "MLP"。
    """
    def __init__(self, num_virtual_tokens=20, reparameterization_type="MLP"):
        """
        初始化 PTuningBert 模型结构。

        Args:
            num_virtual_tokens (int): 插入的虚拟 token 数量。
            reparameterization_type (str): 重参数化网络类型，"MLP" 或 "LSTM"。
        """
        super().__init__()  # 调用父类 nn.Module 的初始化，注册参数和子模块管理机制

        # 加载预训练模型
        # AutoModel.from_pretrained 返回类型: BertModel，包含 embeddings / encoder / pooler 三部分
        self.model = AutoModel.from_pretrained("bert-base-uncased", cache_dir=CACHE_DIR_MODEL/"bert-base-uncased")

        # 二分类线性分类头：将 [CLS] 位置的隐藏向量（维度 hidden_size）映射为标量 logit
        # nn.Linear(in_features, out_features): in=hidden_size(768), out=1
        # 返回类型: nn.Linear，输出形状: (batch_size, 1)
        self.classifier = nn.Linear(self.model.config.hidden_size, 1)

        # 冻结除分类器层之外的参数
        # requires_grad=False 后，反向传播不会为这些参数计算梯度，从而节省显存并保留预训练知识
        for param in self.model.parameters():
            param.requires_grad = False  # 逐一冻结 BERT 所有权重，只有 prompt 和分类头参与更新

        # hidden_size: int，BERT 每一层的隐藏维度，bert-base-uncased 为 768
        hidden_size = self.model.config.hidden_size

        self.num_virtual_tokens = num_virtual_tokens  # 虚拟token数目，决定前缀长度

        # 定义一个可学习的参数，作为虚拟提示的初始值，其形状为(num_virtual_tokens, hidden_size)
        # nn.Parameter 使张量自动注册为模型参数（requires_grad=True），形状: (num_virtual_tokens, 768)
        self.prompt = nn.Parameter(torch.zeros(self.num_virtual_tokens, hidden_size))
        print(self.prompt.shape)  # 打印虚拟提示张量的形状，供调试确认，输出: torch.Size([20, 768])

        # 重新参数化,根据传入的reparameterization_type参数，初始化不同的重新参数化头部
        # 重参数化的作用：对原始可学习 prompt 做非线性变换，增强表达能力，避免直接优化裸向量难以收敛
        self.reparameterization_type = reparameterization_type  # 保存类型供 forward 分支判断

        if reparameterization_type == "MLP":
            # 三层全连接 + ReLU 的 MLP 重参数化头
            # 输入形状: (batch_size, num_virtual_tokens, hidden_size)
            # 输出形状: (batch_size, num_virtual_tokens, hidden_size)，维度不变，但特征经过非线性变换
            self.mlp_head = nn.Sequential(
                nn.Linear(hidden_size, hidden_size),  # 第一层线性变换，维度保持 768→768
                nn.ReLU(),                            # 非线性激活，引入非线性表达
                nn.Linear(hidden_size, hidden_size),  # 第二层线性变换，768→768
                nn.ReLU(),                            # 非线性激活
                nn.Linear(hidden_size, hidden_size),  # 第三层线性变换，768→768
            )
        elif reparameterization_type == "LSTM":
            # 双向 LSTM 重参数化头：对 token 序列建模上下文依赖
            # 输入形状: (batch_size, num_virtual_tokens, hidden_size)
            # 输出形状: (batch_size, num_virtual_tokens, hidden_size * 2)，因双向故翻倍
            self.lstm_head = nn.LSTM(
                input_size=hidden_size,   # 每个时间步的输入维度，与 BERT 隐藏维度对齐，768
                hidden_size=hidden_size,  # LSTM 每个方向的隐藏单元数，768
                num_layers=2,             # 堆叠 2 层 LSTM，增强序列建模能力
                bidirectional=True,       # 双向 LSTM，同时捕捉正向和反向上下文
                batch_first=True,         # 输入张量第 0 维为 batch，即 (batch, seq, feature)
            )
            # LSTM 输出维度为 hidden_size*2（双向拼接），MLP 将其压缩回 hidden_size
            # 输入形状: (batch_size, num_virtual_tokens, hidden_size*2)
            # 输出形状: (batch_size, num_virtual_tokens, hidden_size)
            self.mlp_head = nn.Sequential(
                nn.Linear(hidden_size * 2, hidden_size * 2),  # 第一层，维度保持 1536→1536
                nn.ReLU(),                                     # 非线性激活
                nn.Linear(hidden_size * 2, hidden_size),       # 第二层，压缩维度 1536→768
            )

    def forward(self, input_ids, attention_mask, **args):
        """
        前向传播：拼接虚拟提示与真实输入嵌入，送入 BERT 完成分类。

        Args:
            input_ids     (torch.Tensor): 输入 token id，形状 (batch_size, seq_len)。
            attention_mask(torch.Tensor): 注意力掩码，形状 (batch_size, seq_len)，1 表示有效位置。
            **args: 忽略其他传入参数，保持接口兼容性。

        Returns:
            torch.Tensor: 二分类概率值，形状 (batch_size,)，取值范围 [0, 1]。
        """
        # 获取输入的批次大小
        # input_ids.size(0): int，当前 batch 中样本数量
        batch_size = input_ids.size(0)

        # 将虚拟提示扩展到与输入相同批次大小 (batch_size,num_virtual_tokens,hidden_size)
        # unsqueeze(0): 在第 0 维增加一个维度，形状从 (num_virtual_tokens, 768) → (1, num_virtual_tokens, 768)
        # expand(batch_size, -1, -1): 沿第 0 维复制 batch_size 次，-1 表示该维度保持不变
        # 返回形状: (batch_size, num_virtual_tokens, 768)，共享内存，不额外分配
        prompt = self.prompt.unsqueeze(0).expand(batch_size, -1, -1)

        # 根据选择的重新参数化类型，对虚拟提示进行处理
        if self.reparameterization_type == "MLP":
            prompt = self.mlp_head(prompt)  # MLP 变换后形状不变: (batch_size, num_virtual_tokens, hidden_size)
        elif self.reparameterization_type == "LSTM":
            prompt, _ = self.lstm_head(prompt)  # prompt = output（所有时间步输出），形状: (batch_size, num_virtual_tokens, hidden_size*2)
            # _ = (h_n, c_n)，即最终隐藏状态，此处丢弃不用
            # 注意：此处使用全序列 output 而非 h_n 是有意为之——
            #   双向 LSTM 中 output[t] = [forward(0..t) ; backward(t..end)]，每个位置的表示各不相同，
            #   保留了位置多样性，使每个虚拟 token 获得不同的初始化向量。
            #   若改用 h_n（完整双向句向量）则所有位置会得到相同向量，失去位置区分性，不适合 P-Tuning。
            prompt = self.mlp_head(prompt)       # MLP 将维度从 hidden_size*2 压缩回 hidden_size: (batch_size, num_virtual_tokens, hidden_size)

        # 将虚拟提示与输入的嵌入层输出连接，形成扩展的输入嵌入
        # self.model.embeddings(input_ids): 对 token id 做嵌入查表+位置编码+LayerNorm，返回形状 (batch_size, seq_len, hidden_size)
        embedding_output = self.model.embeddings(input_ids)

        # torch.cat(..., dim=1): 在序列长度维度（dim=1）拼接虚拟提示与真实嵌入
        # 拼接后形状: (batch_size, num_virtual_tokens + seq_len, hidden_size)
        extended_inputs_embeds = torch.cat([prompt, embedding_output], dim=1)  # 在seq_len进行拼接

        # 掩码要同步改变，虚拟token对应的mask值都是1
        # torch.ones(...): 为 num_virtual_tokens 个虚拟 token 生成全 1 掩码（均为有效位置）
        # 拼接后形状: (batch_size, num_virtual_tokens + seq_len)
        # 虚拟提示不需要padding，需要自注意
        extended_attention_mask = torch.cat([
            torch.ones(batch_size, self.num_virtual_tokens).to(input_ids.device),  # 虚拟 token 掩码全为 1，设备与输入一致
            attention_mask  # 原始真实 token 的掩码，形状 (batch_size, seq_len)
        ], dim=1)  # 在序列长度维度拼接，保证掩码与嵌入长度一致

        # 将扩展的输入嵌入和注意力掩码输入BERT模型
        # inputs_embeds 替代 input_ids 直接传入嵌入，绕过 BERT 自带的 embedding 查表层
        # outputs 类型: BaseModelOutputWithPoolingAndCrossAttentions
        outputs = self.model(inputs_embeds=extended_inputs_embeds, attention_mask=extended_attention_mask)

        # 获取经过BERT模型处理后的特定位置的特征，即虚拟提示之后的第一个位置，等价于原来的第0个位置
        # outputs.last_hidden_state 形状: (batch_size, num_virtual_tokens + seq_len, hidden_size)
        # [:, self.num_virtual_tokens, :]: 取拼接后序列中虚拟 token 之后的第一个位置（即原始 [CLS] 对应的输出）
        # feature 形状: (batch_size, hidden_size)
        feature = outputs.last_hidden_state[:, self.num_virtual_tokens, :]

        # 将 [CLS] 位置的特征映射为标量 logit
        # logits 形状: (batch_size, 1)
        logits = self.classifier(feature)

        # torch.sigmoid: 将 logit 映射到 [0, 1] 概率区间
        # .squeeze(): 去除最后的大小为 1 的维度，形状从 (batch_size, 1) → (batch_size,)
        return torch.sigmoid(logits).squeeze()


# 实例化 P-Tuning 模型（默认使用 MLP 重参数化）
ptuning_bert = PTuningBert(reparameterization_type="LSTM")
# ptuning_bert = PTuningBert(reparameterization_type='LSTM')  # 可切换为 LSTM 重参数化

print('-'*50)    # 打印分隔线，便于阅读输出
print(ptuning_bert)  # 打印模型结构，显示各子模块名称与参数规模
print('-'*50)    # 打印分隔线

# 记录参数数量
# def count_parameters(model):
#     return sum(p.numel() for p in model.parameters() if p.requires_grad)

# 统计并打印可训练参数数量，验证只有 prompt 和分类头参与训练
count_parameters(ptuning_bert)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([20, 768])
--------------------------------------------------
PTuningBert(
  (model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768,

In [ ]:
# 打印 P-Tuning 模型中所有参数的名称和形状，验证哪些参数可训练
# 预期结果：只有 prompt（虚拟 token 嵌入）和 mlp_head 的参数出现
# 因为 self.model 的所有参数已被冻结（requires_grad=False）
for idx, (key, value) in enumerate(ptuning_bert.named_parameters()):
    # key   类型: str，参数名路径（如 "prompt"、"mlp_head.0.weight"）
    # value 类型: nn.Parameter，形状通过 value.shape 查看
    print(f'key:{key}--value:{value.shape}')

key:prompt--value:torch.Size([20, 768])
key:model.embeddings.word_embeddings.weight--value:torch.Size([30522, 768])
key:model.embeddings.position_embeddings.weight--value:torch.Size([512, 768])
key:model.embeddings.token_type_embeddings.weight--value:torch.Size([2, 768])
key:model.embeddings.LayerNorm.weight--value:torch.Size([768])
key:model.embeddings.LayerNorm.bias--value:torch.Size([768])
key:model.encoder.layer.0.attention.self.query.weight--value:torch.Size([768, 768])
key:model.encoder.layer.0.attention.self.query.bias--value:torch.Size([768])
key:model.encoder.layer.0.attention.self.key.weight--value:torch.Size([768, 768])
key:model.encoder.layer.0.attention.self.key.bias--value:torch.Size([768])
key:model.encoder.layer.0.attention.self.value.weight--value:torch.Size([768, 768])
key:model.encoder.layer.0.attention.self.value.bias--value:torch.Size([768])
key:model.encoder.layer.0.attention.output.dense.weight--value:torch.Size([768, 768])
key:model.encoder.layer.0.attention.out

In [ ]:
# 训练 P-Tuning 模型（仅 prompt + MLP 重参数化头可训练）并记录结果
training_record["P-Tuning"] = train(
    ptuning_bert, train_loader, val_loader, device,
    num_epochs=num_epochs, patience=patience
)

In [ ]:
# 释放 P-Tuning 模型占用的 GPU 显存
del ptuning_bert

In [ ]:
# 更新对比图，加入 P-Tuning 方法的验证准确率曲线
plot_training_record(training_record, metric_name="val_acc")

### 5.2 Prefix Tuning（各层注意力前缀注入）

**思路**：不只在输入嵌入层插入软提示，而是在 BERT **每一层** Transformer 的多头注意力中都注入可学习的前缀键值对（past_key_values），通过 MLP 投影生成各层前缀，让软提示的影响渗透到模型的每一层，同时冻结 BERT 全部参数。

与 P-Tuning v2 的区别：Prefix Tuning 用**一个共享 MLP** 从同一组隐向量出发生成所有层的前缀（各层参数由 MLP 耦合，不完全独立）；P-Tuning v2（原论文）则为**每一层独立维护**一组可学习的前缀参数，直接优化、互不干扰，表达能力更强。

In [ ]:
# 定义 Prefix Tuning BERT 模型：在每一层 Transformer 的注意力中注入可学习的前缀键值对
class PrefixTuningBert(nn.Module):
    """
    Prefix Tuning BERT：通过向每层注意力传入 past_key_values 的方式注入可学习前缀。

    与 P-Tuning 不同，Prefix Tuning 不在嵌入层拼接，而是直接将前缀作为每层注意力的
    past_key_values 传入，从而在所有 Transformer 层都产生引导效果。

    Args:
        num_virtual_tokens  (int) : 前缀虚拟 token 数量，默认 20。
        prefix_projection   (bool): 是否通过 MLP 对前缀做非线性重参数化，默认 True。
    """
    def __init__(self, num_virtual_tokens=20, prefix_projection=True):
        """
        初始化 PrefixTuningBert 模型。

        Args:
            num_virtual_tokens  (int) : 前缀虚拟 token 数量。
            prefix_projection   (bool): True 则用 MLP 对前缀做非线性映射；False 则直接使用原始向量。
        """
        super().__init__()  # 调用父类 nn.Module 的初始化方法

        # 加载预训练的 BERT 模型
        # self.model 类型: transformers.BertModel，提供 embeddings / encoder / pooler
        self.model = AutoModel.from_pretrained("bert-base-uncased", cache_dir=CACHE_DIR_MODEL/"bert-base-uncased")

        # 线性分类头：将 [CLS] token 的 hidden_size 维特征映射为 1 个标量 logit
        # 输入形状: (batch_size, hidden_size=768)，输出形状: (batch_size, 1)
        self.classifier = nn.Linear(self.model.config.hidden_size, 1)

        # 冻结预训练模型的所有参数，只有前缀参数和分类头参与训练
        for param in self.model.parameters():
            param.requires_grad = False  # 冻结：反向传播不计算该参数的梯度

        # ── 前缀相关超参数 ────────────────────────────────────────────────────
        self.num_virtual_tokens = num_virtual_tokens              # 前缀虚拟 token 数量（int）
        self.prefix_projection = prefix_projection                # 是否使用 MLP 重参数化（bool）
        hidden_size = self.model.config.hidden_size               # BERT 隐藏维度，bert-base 为 768（int）
        self.num_layers = self.model.config.num_hidden_layers     # Transformer 编码器层数，bert-base 为 12（int）
        self.num_attention_heads = self.model.config.num_attention_heads  # 多头注意力的头数，bert-base 为 12（int）
        # 每个注意力头的维度 = hidden_size // num_attention_heads = 768 // 12 = 64（int）
        self.embed_size_per_head = hidden_size // self.num_attention_heads

        if self.prefix_projection:
            # 使用带 MLP 重参数化的前缀：原始参数形状为 (num_virtual_tokens, hidden_size)
            # 再通过两层 MLP 映射到每层所需的 key/value 空间，增强表达能力防止直接优化裸向量难收敛
            # prefix_tokens 类型: nn.Parameter，形状: (num_virtual_tokens, hidden_size)，可学习
            self.prefix_tokens = nn.Parameter(torch.zeros(self.num_virtual_tokens, hidden_size))
            # MLP 投影网络：将 hidden_size → hidden_size → num_layers × 2 × hidden_size
            # 输出维度 num_layers × 2 × hidden_size 代表：12 层 × (key + value) × 768 = 18432
            self.transform = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_size),                          # 第一层：768→768
                torch.nn.Tanh(),                                                    # Tanh 激活，值域 (-1, 1)
                torch.nn.Linear(hidden_size, self.num_layers * 2 * hidden_size),    # 第二层：768→18432
            )
        else:
            # 不使用前缀投影，直接将原始向量作为 past_key_values（维度需手动对齐）
            # prefix_tokens 类型: nn.Parameter，形状: (num_virtual_tokens, hidden_size)
            self.prefix_tokens = nn.Parameter(torch.zeros(self.num_virtual_tokens, hidden_size))

    def forward(self, input_ids, attention_mask, **args):
        """
        前向传播：构造 past_key_values 并注入 BERT 每层注意力，完成分类。

        Args:
            input_ids      (torch.Tensor): token id 序列，形状 (batch_size, seq_len)。
            attention_mask (torch.Tensor): 注意力掩码，形状 (batch_size, seq_len)。
            **args: 接收并忽略其余关键字参数，保持接口兼容性。

        Returns:
            torch.Tensor: 二分类概率值，形状 (batch_size,)，值域 [0, 1]。
        """
        batch_size = input_ids.size(0)  # 获得批次大小（int）

        # 将前缀 token 扩展到批次维度
        # unsqueeze(0): (num_virtual_tokens, 768) → (1, num_virtual_tokens, 768)
        # expand: 沿 batch 维复制 batch_size 次（共享内存），形状: (batch_size, num_virtual_tokens, 768)
        prefix_tokens = self.prefix_tokens.unsqueeze(0).expand(batch_size, -1, -1)

        if self.prefix_projection:
            # MLP 投影：将 (batch_size, num_virtual_tokens, 768) → (batch_size, num_virtual_tokens, num_layers*2*hidden_size)
            # 即 (batch_size, 20, 18432)，包含所有层的 key 和 value 拼接
            past_key_values = self.transform(prefix_tokens)
        else:
            # 不投影时直接使用原始向量，形状: (batch_size, num_virtual_tokens, hidden_size)
            past_key_values = prefix_tokens

        # ── 将扁平化的前缀向量重塑为 BERT 所需的 past_key_values 格式 ──────────────
        # 目标形状: (batch_size, num_layers, 2, num_attention_heads, num_virtual_tokens, embed_size_per_head)
        # 其中 2 代表 key 和 value 两组矩阵
        past_key_values = past_key_values.view(
            batch_size, self.num_layers, 2, self.num_attention_heads, -1, self.embed_size_per_head
        )  # 形状变为: (batch_size, 12, 2, 12, 20, 64)

        # permute: 调整维度顺序，将 key/value 维度置于最前
        # 变换后形状: (2, num_layers, batch_size, num_attention_heads, num_virtual_tokens, embed_size_per_head)
        past_key_values = past_key_values.permute(2, 1, 0, 3, 4, 5)

        # 构造 BERT 所需的 past_key_values 格式：
        # 外层 tuple 长度 = num_layers（12），每个元素为 (key, value) 元组
        # key/value 形状: (batch_size, num_attention_heads, num_virtual_tokens, embed_size_per_head)
        #              即 (batch_size, 12, 20, 64)
        past_key_values = tuple([
            tuple([past_key_values[0][i], past_key_values[1][i]])  # (key_i, value_i) 第 i 层
            for i in range(self.num_layers)
        ])

        # 扩展注意力掩码：为前缀 token 补充全 1 掩码（前缀位置均有效，不掩盖）
        # 拼接后形状: (batch_size, num_virtual_tokens + seq_len)
        extended_attention_mask = torch.cat([
            torch.ones(batch_size, self.num_virtual_tokens).to(attention_mask.device),  # 前缀掩码全为 1
            attention_mask  # 原始序列掩码，形状 (batch_size, seq_len)
        ], dim=1)

        # 将 input_ids、扩展掩码及 past_key_values 输入 BERT
        # past_key_values 是 HuggingFace BertModel.forward() 的官方参数（默认为 None）：
        #   HuggingFace 对所有 Transformer 模型统一了接口，将其作为通用参数加入，
        #   BERT 本身不做自回归生成，普通推理时无需传入，此处借用来注入可学习前缀。
        #   BERT 每层 Self-Attention 会将传入的前缀 K/V 拼接到序列 K/V 前面，
        #   使每个真实 token 的注意力都能「看到」前缀向量，受其引导，无需修改 BERT 源码。
        #   格式要求: tuple of tuple，长度=num_layers，每个内层 tuple=(key, value)，
        #   key/value 形状: (batch_size, num_heads, num_virtual_tokens, head_dim)
        # outputs 类型: BaseModelOutputWithPoolingAndCrossAttentions
        outputs = self.model(input_ids, extended_attention_mask, past_key_values=past_key_values)

        # 取最后一层 [CLS] token（位置 0）的隐藏状态作为句子级特征
        # feature 形状: (batch_size, hidden_size=768)
        feature = outputs.last_hidden_state[:, 0, :]

        logits = self.classifier(feature)         # 线性映射为标量 logit，形状: (batch_size, 1)
        return torch.sigmoid(logits).squeeze()    # sigmoid + squeeze → 形状: (batch_size,)，值域 [0,1]


# 实例化前缀调优 BERT 模型（默认启用 MLP 重参数化）
prefix_tuning_bert = PrefixTuningBert()
# 打印模型结构（可选，层级较深时输出较长）
# print(prefix_tuning_bert)
# 打印参数统计：可训练参数为 prefix_tokens + transform MLP，约占总参数的 0.14%
count_parameters(prefix_tuning_bert)



In [ ]:
# 计算 Prefix Tuning 中 past_key_values 变换前的中间维度验证
# 每层注意力需要为前缀 token 提供 key 和 value 两个矩阵
# hidden_size=768，num_layers=12，2 代表 key 和 value
# 结果: 18432，即 MLP 第二层输出维度（num_layers × 2 × hidden_size）
768 * 12 * 2

In [ ]:
# 验证注意力头数 × 每头维度 = 隐藏层维度
# num_attention_heads=12，embed_size_per_head = hidden_size // num_attention_heads = 768 // 12 = 64
# 结果: 768，与 hidden_size 一致，验证维度拆分正确
12 * 64

In [ ]:
# 训练 Prefix Tuning 模型（可训练参数：前缀 token 参数 + MLP 投影层）
training_record["Prefix Tuning"] = train(
    prefix_tuning_bert, train_loader, val_loader, device,
    num_epochs=num_epochs, patience=patience
)

In [ ]:
# 释放 Prefix Tuning 模型的 GPU 显存
del prefix_tuning_bert

In [ ]:
# 将 HuggingFace 缓存中 BERT 模型文件复制到当前目录（可选，用于离线环境）
# 取消注释后运行，将模型文件复制到 MyDrive，下次可从本地加载无需重新下载
# !cp -r ~/.cache/huggingface/hub/models--bert-base-uncased .

In [ ]:
# 打印当前工作目录（Print Working Directory），确认文件读写路径
# 输出类型: str（通过 Shell 打印），预期为 /content/drive/MyDrive/ 或本地路径
!pwd

In [ ]:
# 更新对比图，加入 Prefix Tuning 方法的验证准确率曲线
plot_training_record(training_record, metric_name="val_acc")

### 5.3 P-Tuning v2（深层全层前缀注入）

**思路**：在 Prefix Tuning 的基础上进一步改进，在 BERT 每一层注意力中注入可学习的前缀键值对，同时冻结 BERT 全部权重。本实现同样使用 MLP 做重参数化以提升训练稳定性，与 Prefix Tuning 的主要区别在于：前缀嵌入改用 `nn.Embedding`（均匀分布初始化）替代 `nn.Parameter`（全零初始化），收敛更快。原论文的 P-Tuning v2 主张各层直接独立优化、去掉 MLP，但工程实现中保留 MLP 是常见做法。

In [ ]:
import torch                          # 导入 PyTorch 核心库
import torch.nn as nn                  # 导入神经网络模块
from transformers import AutoModel, AutoTokenizer  # 自动加载预训练模型和分词器

class PTuningV2Bert(nn.Module):
    """
    P-Tuning v2 模型：在 BERT 每一层的注意力中均注入可学习前缀（past_key_values），
    相比 P-Tuning（仅修改输入嵌入层）可学习参数更多，对模型的引导更深入、更直接。

    与 Prefix Tuning 的主要区别：前缀嵌入使用 nn.Embedding（均匀分布初始化）
    替代 nn.Parameter（全零初始化），收敛更稳定。两者均使用 MLP 做重参数化；
    原论文 P-Tuning v2 主张去掉 MLP、各层独立直接优化，本实现保留 MLP 以提升训练稳定性。

    Args:
        num_virtual_tokens  (int) : 前缀虚拟 token 数量，默认 20。
        prefix_projection   (bool): 是否使用 MLP 对前缀做非线性重参数化，默认 True。
    """
    def __init__(self, num_virtual_tokens=20, prefix_projection=True):
        """
        初始化 PTuningV2Bert 模型。

        Args:
            num_virtual_tokens  (int) : 前缀虚拟 token 数量。
            prefix_projection   (bool): True 则对前缀做 MLP 重参数化映射，False 则直接使用。
        """
        super().__init__()  # 调用父类 nn.Module 的初始化，注册子模块管理机制

        # 加载预训练的 BERT 模型
        # self.model 类型: transformers.BertModel，含 embeddings / encoder / pooler
        self.model = AutoModel.from_pretrained("bert-base-uncased", cache_dir=CACHE_DIR_MODEL/"bert-base-uncased")

        # 线性分类头：将 [CLS] token 的 768 维特征映射为 1 个输出 logit
        # 输出形状: (batch_size, 1)
        self.classifier = nn.Linear(self.model.config.hidden_size, 1)

        # 冻结预训练模型所有参数，只有前缀和分类头参与训练
        for param in self.model.parameters():
            param.requires_grad = False  # 冻结：不参与反向传播梯度计算

        # ── 前缀超参数 ───────────────────────────────────────────────────────
        self.num_virtual_tokens = num_virtual_tokens              # 前缀虚拟 token 数量（int）
        self.prefix_projection = prefix_projection                # 是否使用 MLP 重参数化（bool）
        hidden_size = self.model.config.hidden_size               # BERT 隐藏维度，768（int）
        self.num_layers = self.model.config.num_hidden_layers     # Transformer 层数，12（int）
        self.num_attention_heads = self.model.config.num_attention_heads  # 注意力头数，12（int）
        # 每个注意力头的维度 = 768 // 12 = 64（int）
        self.embed_size_per_head = hidden_size // self.num_attention_heads

        # P-Tuning v2 改进点：使用 nn.Embedding 替代 nn.Parameter，
        # 支持均匀分布初始化，相比全零初始化收敛更快
        # prefix_embeddings 类型: nn.Embedding，权重形状: (num_virtual_tokens, hidden_size)
        self.prefix_embeddings = nn.Embedding(self.num_virtual_tokens, hidden_size)
        # 均匀分布初始化：权重在 [-0.1, 0.1] 均匀采样，比全零初始化有更好的多样性起点
        nn.init.uniform_(self.prefix_embeddings.weight, -0.1, 0.1)

        if self.prefix_projection:
            # MLP 投影层：hidden_size → hidden_size → num_layers × 2 × hidden_size
            # 输出维度 = 12 × 2 × 768 = 18432，包含所有层的 key/value 向量
            self.prefix_projection_layer = nn.Sequential(
                nn.Linear(hidden_size, hidden_size),                         # 第一层：768→768
                nn.Tanh(),                                                   # Tanh 激活，值域 (-1, 1)
                nn.Linear(hidden_size, self.num_layers * 2 * hidden_size)    # 第二层：768→18432
            )

    def forward(self, input_ids, attention_mask, **args):
        """
        前向传播：构造每层的 past_key_values 前缀并注入 BERT，完成句子分类。

        Args:
            input_ids      (torch.Tensor): token id，形状 (batch_size, seq_len)。
            attention_mask (torch.Tensor): 注意力掩码，形状 (batch_size, seq_len)。
            **args: 接收并忽略其余参数，保持接口兼容性。

        Returns:
            torch.Tensor: 正类概率，形状 (batch_size,)，值域 [0, 1]。
        """
        batch_size = input_ids.size(0)  # 当前批次大小（int）

        # 生成前缀 token 索引序列：[0, 1, ..., num_virtual_tokens-1]，扩展到批次维度
        # torch.arange: 形状 (num_virtual_tokens,)，各元素为嵌入表的索引
        # unsqueeze(0): → (1, num_virtual_tokens)
        # expand: → (batch_size, num_virtual_tokens)
        prefix_tokens = torch.arange(
            self.num_virtual_tokens, device=input_ids.device
        ).unsqueeze(0).expand(batch_size, -1)

        # 通过嵌入表查找前缀向量
        # prefix_embeddings 形状: (batch_size, num_virtual_tokens, hidden_size=768)
        prefix_embeddings = self.prefix_embeddings(prefix_tokens)

        if self.prefix_projection:
            # MLP 投影：(batch_size, num_virtual_tokens, 768) → (batch_size, num_virtual_tokens, 18432)
            past_key_values = self.prefix_projection_layer(prefix_embeddings)
        else:
            # 不投影时直接用原始嵌入，形状: (batch_size, num_virtual_tokens, hidden_size)
            past_key_values = prefix_embeddings

        # ── 将扁平向量重塑为 BERT past_key_values 所需的格式 ─────────────────
        # 目标: (batch_size, num_layers, 2, num_attention_heads, num_virtual_tokens, embed_size_per_head)
        # 其中 2 代表每层需要 key 和 value 两组前缀矩阵（K/V 各一份）
        past_key_values = past_key_values.view(
            batch_size, self.num_layers, 2, self.num_attention_heads, -1, self.embed_size_per_head
        )  # 形状: (batch_size, 12, 2, 12, 20, 64)

        # permute: 将 key/value 维度调整到最前，与 BERT 接口对齐
        # 形状变为: (2, num_layers, batch_size, num_attention_heads, num_virtual_tokens, embed_size_per_head)
        past_key_values = past_key_values.permute(2, 1, 0, 3, 4, 5)

        # 构造 BERT 要求的 tuple[tuple[Tensor, Tensor]] 格式：
        # 外层 tuple 长度 = num_layers；每个内层 tuple 为 (key_i, value_i)
        # key_i / value_i 形状: (batch_size, num_attention_heads, num_virtual_tokens, embed_size_per_head)
        past_key_values = tuple([
            tuple([past_key_values[0][i], past_key_values[1][i]])  # 第 i 层的 (key, value)
            for i in range(self.num_layers)
        ])

        # 扩展注意力掩码：前缀 token 的掩码全为 1（均参与注意力），与原始掩码拼接
        # 拼接后形状: (batch_size, num_virtual_tokens + seq_len)
        extended_attention_mask = torch.cat([
            torch.ones(batch_size, self.num_virtual_tokens).to(attention_mask.device),  # 前缀掩码全 1
            attention_mask  # 原始序列掩码，形状 (batch_size, seq_len)
        ], dim=1)

        # 将 BERT 的输入和前缀 past_key_values 传入模型
        # past_key_values 是 HuggingFace BertModel.forward() 的官方参数（默认为 None）：
        #   HuggingFace 对所有 Transformer 模型统一了接口，将其作为通用参数加入，
        #   BERT 本身不做自回归生成，普通推理时无需传入，此处借用来注入可学习前缀。
        #   BERT 每层 Self-Attention 会将传入的前缀 K/V 拼接到序列 K/V 前面，
        #   使每个真实 token 的注意力都能「看到」前缀向量，受其引导，无需修改 BERT 源码。
        #   格式要求: tuple of tuple，长度=num_layers，每个内层 tuple=(key, value)，
        #   key/value 形状: (batch_size, num_heads, num_virtual_tokens, head_dim)
        # attention_mask 用 keyword 形式传入，避免与位置参数混淆
        # outputs 类型: BaseModelOutputWithPoolingAndCrossAttentions
        outputs = self.model(input_ids, attention_mask=extended_attention_mask, past_key_values=past_key_values)

        # 取 [CLS] token（位置 0）的最后一层隐藏状态作为句子级特征
        # feature 形状: (batch_size, hidden_size=768)
        feature = outputs.last_hidden_state[:, 0, :]

        logits = self.classifier(feature)         # 线性分类，形状: (batch_size, 1)
        return torch.sigmoid(logits).squeeze()    # sigmoid + squeeze，形状: (batch_size,)，值域 [0,1]


# 实例化 P-Tuning v2 模型（默认启用 MLP 重参数化）
p_tuning_v2_bert = PTuningV2Bert()
print(p_tuning_v2_bert)   # 打印模型结构，查看各子模块

# 打印参数统计（可训练参数：prefix_embeddings + prefix_projection_layer）
# count_parameters 内部已打印结果，返回 None
# def count_parameters(model):
#     return sum(p.numel() for p in model.parameters() if p.requires_grad)
count_parameters(p_tuning_v2_bert)

# 训练 P-Tuning v2 模型并记录验证指标到 training_record
training_record["P-Tuning v2"] = train(p_tuning_v2_bert, train_loader, val_loader, device, num_epochs=num_epochs, patience=patience)
del p_tuning_v2_bert


In [ ]:
# 更新对比图，加入 P-Tuning v2 方法的验证准确率曲线
plot_training_record(training_record, metric_name="val_acc")

## 六、LoRA低秩适配（Low-Rank Adaptation）

**论文**：LoRA: LOW-RANK ADAPTATION OF LARGE LANGUAGE MODELS

**核心思想**：通过低秩分解来模拟参数的增量矩阵 ΔW，从而以极小的参数量实现大模型的间接训练。

在涉及矩阵乘法的模块（如注意力中的 Q、K、V、O 投影）旁边增加一条新的并行通路：
- 矩阵 **A**（降维）：将输入从 d 维映射到低秩维度 r（d >> r）
- 矩阵 **B**（升维）：将 r 维特征映射回 d 维输出空间
- 最终输出 = 原始冻结矩阵输出 + **B(A(x)) × scaling**，其中 scaling = lora_alpha / rank

B 初始化为 0，使得训练初期 LoRA 输出为 0，不破坏预训练权重的初始状态。

**设计思想**：微调时权重的变化量 ΔW 本质上是低秩的——真正有用的更新方向只有少数几个。LoRA 用 $\Delta W = A \times B$（$r \ll d$）来参数化这一低维子空间，冻结原始权重 $W_0$，只训练 A、B 两个小矩阵。训练完成后可将 AB 直接合并回 $W_0$，推理无额外开销。这与提示调优系列（P-Tuning / Prefix Tuning）在注意力中注入额外 token 的方式不同——后者是"引导"模型利用已有能力，而 LoRA 是直接修改权重矩阵本身，梯度路径更短、更新信号更强，因此效果更接近全量微调。

rank的作用：
rank代表低秩矩阵的秩，即线性变换矩阵A和B的输出特征数量。在LoRA中，原始的高维特征通过线性变换A被映射到一个低维空间（rank维），然后再通过另一个线性变换B映射回原始特征空间。较低的rank值意味着更少的参数需要更新，从而降低了模型复杂度和计算成本。

lora_alpha的作用：
lora_alpha是一个缩放因子，用于调整LoRA输出的贡献。它通过除以rank来计算得到scaling，这个缩放因子被用于控制低秩空间中的特征对最终输出的贡献度。较大的lora_alpha值会增加LoRA特征的影响力，而较小的值则会减少其影响。

In [ ]:
# 遍历 BERT 编码器的所有 Transformer 层，打印每层的类型
# bert_model.encoder.layer 类型: nn.ModuleList，共 12 个 BertLayer 对象
# 通过此遍历可以确认每层的类名，为后续 LoRA 替换目标层做准备
# type(layer) 返回: <class 'transformers.models.bert.modeling_bert.BertLayer'>
for layer in bert_model.encoder.layer:
    print(type(layer))   # 打印当前层的类型（应为 BertLayer）
    print('-' * 50)      # 分隔线，便于区分各层输出

In [ ]:
import torch                          # PyTorch 核心库
import torch.nn as nn                  # 神经网络模块（Linear、Module 等）
from transformers import AutoModel     # 自动加载预训练模型

class LoRALayer(nn.Module):
    """
    LoRA 低秩适配层：用低秩矩阵 A、B 旁路原始线性层，实现参数高效微调。

    前向传播公式：
        output = frozen_linear(x) + B(A(x)) * scaling

    原始线性层权重冻结（no_grad），只有 A、B 两个小矩阵参与训练。
    B 初始化为零，保证训练初期 LoRA 旁路输出为 0，不破坏预训练初始状态。

    Args:
        module    (nn.Linear): 需要插入 LoRA 的原始线性层（会被冻结）。
        rank      (int)      : 低秩维度 r，r 越小参数越少，须为 > 1 的正整数，默认 8。
        lora_alpha(int)      : 缩放系数分子，scaling = lora_alpha / rank，默认 1。
    """
    def __init__(self, module: nn.Module, rank: int = 8, lora_alpha: int = 1):
        super().__init__()  # 调用父类初始化，注册子模块

        # 校验 rank 必须为 > 1 的正整数（过小的 rank 几乎无表达能力）
        assert isinstance(rank, int) and rank > 1, "Lora rank should be a positive integer"

        # 缩放因子：scaling = lora_alpha / rank（float）
        # 参考论文，固定 lora_alpha 而改变 rank 时，等价于调整学习率，scaling 使二者解耦
        self.scaling = lora_alpha / rank

        self.module = module  # 保存原始冻结线性层（nn.Linear），forward 中以 no_grad 调用

        # 降维矩阵 A：将输入从 in_features 维映射到低秩维度 r
        # 类型: nn.Linear，权重形状: (rank, in_features)，bias=False
        self.A = nn.Linear(module.in_features, rank, bias=False)

        # 升维矩阵 B：将低秩维度 r 映射回 out_features
        # 类型: nn.Linear，权重形状: (out_features, rank)，bias=False
        self.B = nn.Linear(rank, module.out_features, bias=False)

        # Kaiming 均匀初始化 A 的权重（a=sqrt(5) 对应默认 fan_in 模式）
        # 保证 A 初始有一定分布，使得 forward 开始时 A(x) 非零
        nn.init.kaiming_uniform_(self.A.weight, a=5 ** 0.5)

        # B 权重初始化为全零（LoRA 论文核心设计）
        # 原因：训练开始时 LoRA 旁路输出 = B(A(x)) * scaling = 0（因为 B=0）
        # 因此 forward 输出 = frozen_linear(x) + 0 = 预训练模型的原始输出
        # 这保证了微调的起点与预训练模型完全一致，不会因随机初始化的 B 引入噪声干扰
        # 若 B 也随机初始化，则训练开始时模型输出就已偏离预训练状态，破坏了预训练知识
        # A 不能也初始化为零（否则梯度始终为 0，A/B 均无法更新），故 A 用 Kaiming 初始化
        nn.init.zeros_(self.B.weight)

        # 将 A、B 移动到与原始层权重相同的设备（CPU / GPU）
        self.A.to(device=module.weight.device)
        self.B.to(device=module.weight.device)

    def forward(self, inputs, *args, **kwargs):
        """
        前向传播：原始层输出（frozen）+ LoRA 旁路输出 × scaling。

        Args:
            inputs (torch.Tensor): 输入张量，形状 (batch_size, ..., in_features)。
            *args, **kwargs      : 传递给原始线性层的其余参数（如 bias 等）。

        Returns:
            torch.Tensor: 与原始线性层输出形状相同，形状 (batch_size, ..., out_features)。
        """
        # torch.no_grad() 作用：禁止在此上下文内构建计算图，不保存中间激活值
        # 效果：节省显存 + 加速前向传播（无需为反向传播缓存中间结果）
        # 注意1：no_grad 与 model.eval() 是完全不同的两件事，互相正交：
        #   - torch.no_grad()  → 只影响梯度计算，Dropout/BN 行为不变（仍受 train/eval 模式控制）
        #   - model.eval()     → 只切换 Dropout（关闭）、BatchNorm（用运行统计量）等层的行为，不影响梯度
        # 注意2：此处 self.module 的参数已设置 requires_grad=False，no_grad 是额外的显存优化：
        #   - 若无 no_grad，self.module(inputs) 仍会构建计算图（因为 inputs 可能 requires_grad=True）
        #   - 加上 no_grad 后，冻结层的前向传播完全不进入计算图，显著节省显存
        with torch.no_grad():
            outputs = self.module(inputs, *args, **kwargs)  # 形状: (batch_size, ..., out_features)

        # LoRA 旁路：A 先降维，B 再升维，乘以 scaling 控制贡献幅度
        # self.B(self.A(inputs)) 形状: (batch_size, ..., out_features)
        # 最终输出 = 原始冻结权重的输出 + LoRA 增量输出
        return outputs + self.B(self.A(inputs)) * self.scaling


class LoRABert(nn.Module):
    """
    LoRA BERT 分类器：将 BERT 每层注意力中的 query 和 key 线性层替换为 LoRALayer，
    冻结其余所有参数，只训练 LoRA 矩阵 A/B 和分类头。

    可训练参数量 ≈ num_layers × 2（query+key）× 2（A+B）× rank × hidden_size 的线性组合
    远小于 BERT 全部参数（约 110M）。

    Args:
        rank      (int): LoRA 低秩维度，默认 8。
        lora_alpha(int): LoRA 缩放系数分子，默认 32（scaling = 32/8 = 4）。
    """
    def __init__(self, rank=8, lora_alpha=32):
        super().__init__()  # 调用父类初始化

        # 加载预训练 BERT-base-uncased
        # self.model 类型: transformers.BertModel
        self.model = AutoModel.from_pretrained("bert-base-uncased", cache_dir=CACHE_DIR_MODEL/"bert-base-uncased")

        # 应用 LoRA：将每层注意力的 query、key 线性层替换为 LoRALayer，并冻结其余参数
        self._apply_lora(rank=rank, lora_alpha=lora_alpha)

        # 线性分类头：将 [CLS] 的 768 维特征映射为 1 个 logit
        # 分类头不被 _apply_lora 冻结，也参与训练
        self.classifier = nn.Linear(self.model.config.hidden_size, 1)

    def _apply_lora(self, rank, lora_alpha):
        """
        冻结 BERT 所有参数，并将每层注意力中的 query、key 线性层替换为 LoRALayer。

        Args:
            rank      (int): LoRA 低秩维度。
            lora_alpha(int): LoRA 缩放系数分子。
        """
        # 先冻结所有参数，再对 LoRALayer 内的 A、B 解冻（LoRALayer 初始化时默认 requires_grad=True）
        for param in self.model.parameters():
            param.requires_grad = False  # 冻结全部 BERT 参数

        # 遍历 BERT 编码器所有 12 层，将 query 和 key 投影层替换为 LoRALayer
        # layer.attention.self.query / key 类型: nn.Linear，形状: (hidden_size, hidden_size) = (768, 768)
        # 替换后这两层的 A、B 参数为可学习，原始权重冻结
        # 注：也可替换 value（V）和输出投影（O），这是超参数选择
        # children() 是 nn.Module 的方法，返回当前模块直接子模块的迭代器（不递归）
        # self.model.encoder.layer 类型: nn.ModuleList，包含 12 个 BertLayer 子模块
        # 每次迭代得到一个 BertLayer 对象（含 attention / intermediate / output 三个子模块）
        for layer in self.model.encoder.layer.children():
            # 访问路径：BertLayer.attention（BertAttention）.self（BertSelfAttention）.query/key（nn.Linear）
            # 注意：此处 .self 并非 Python 关键字，而是 HuggingFace BertAttention 中的子模块属性名
            # HuggingFace 源码将 BertSelfAttention 实例命名为 self，对应论文中的 "self-attention" 概念
            # 完整模块层级: BertLayer → BertAttention.self(BertSelfAttention) → query/key/value(nn.Linear)
            layer.attention.self.query = LoRALayer(layer.attention.self.query, rank=rank, lora_alpha=lora_alpha)
            layer.attention.self.key   = LoRALayer(layer.attention.self.key,   rank=rank, lora_alpha=lora_alpha)

    def forward(self, input_ids, attention_mask, token_type_ids):
        """
        前向传播：通过 LoRA-BERT 提取 [CLS] 特征，经分类头输出概率。

        Args:
            input_ids      (torch.Tensor): token id，形状 (batch_size, seq_len)。
            attention_mask (torch.Tensor): 注意力掩码，形状 (batch_size, seq_len)。
            token_type_ids (torch.Tensor): 句子类型 id，形状 (batch_size, seq_len)，单句任务全 0。

        Returns:
            torch.Tensor: 正类概率，形状 (batch_size,)，值域 [0, 1]。
        """
        # 通过 BERT 前向传播（query/key 层内部自动走 LoRA 旁路）
        # outputs 类型: BaseModelOutputWithPoolingAndCrossAttentions
        outputs = self.model(input_ids, attention_mask, token_type_ids)

        # 取最后一层 [CLS] token（位置 0）的隐藏状态，形状: (batch_size, hidden_size=768)
        feature = outputs.last_hidden_state[:, 0, :]

        logits = self.classifier(feature)         # 线性映射，形状: (batch_size, 1)
        return torch.sigmoid(logits).squeeze()    # sigmoid + squeeze，形状: (batch_size,)


# 实例化 LoRA BERT 模型（rank=8，lora_alpha=32，scaling=4）
lora_bert = LoRABert(rank=8, lora_alpha=32)
print(lora_bert)             # 打印模型结构，观察 query/key 已被替换为 LoRALayer
count_parameters(lora_bert)  # 打印参数统计，验证可训练参数远小于全量微调



In [ ]:
# 训练 LoRA 模型（可训练参数：LoRA A/B 矩阵 + 分类头，约占总参数量的 0.3%）
training_record["LoRA"] = train(
    lora_bert, train_loader, val_loader, device,
    num_epochs=num_epochs, patience=patience
)

In [ ]:
# 释放 LoRA 模型占用的 GPU 显存
del lora_bert

In [ ]:
# 更新对比图，加入 LoRA 方法的验证准确率曲线
plot_training_record(training_record, metric_name="val_acc")

## 七、Adapter Tuning适配器微调

**论文**：Parameter-Efficient Transfer Learning for NLP

**核心思想**：在 Transformer 每层的多头注意力投影输出和 FFN 输出后插入 **Adapter 模块**（瓶颈结构：down_project → 非线性激活 → up_project + 残差连接）。训练时冻结原始预训练权重，只训练 Adapter 和 LayerNorm 参数，保证训练高效性。Adapter 的 `adapter_size` 控制瓶颈维度，越小则参数越少。

In [ ]:
class AdapterLayer(nn.Module):
    """
    Adapter 瓶颈模块：down_project → 非线性激活 → up_project + 残差连接。

    通过先降维再升维的瓶颈结构，以极少的参数量（adapter_size << input_size）
    实现对当前层特征的任务适配，同时残差连接保证不破坏原有特征。

    Args:
        input_size   (int): 输入（和输出）特征维度，通常等于 BERT hidden_size（768）。
        adapter_size (int): 瓶颈维度，越小参数越少，通常取 64 或 128。
    """
    def __init__(self, input_size, adapter_size):
        super().__init__()  # 调用父类初始化

        # 降维线性层：将 input_size → adapter_size，大幅压缩特征维度
        # 类型: nn.Linear，权重形状: (adapter_size, input_size)
        self.down_project = nn.Linear(input_size, adapter_size)

        # 非线性激活：ReLU，引入非线性表达能力
        self.nolinearity = nn.ReLU()

        # 升维线性层：将 adapter_size → input_size，恢复原始维度以便残差相加
        # 类型: nn.Linear，权重形状: (input_size, adapter_size)
        self.up_project = nn.Linear(adapter_size, input_size)

    def forward(self, x):
        """
        前向传播：x → down_project → ReLU → up_project，不含残差（残差在外层添加）。

        Args:
            x (torch.Tensor): 输入特征，形状 (batch_size, seq_len, input_size)。

        Returns:
            torch.Tensor: 经 Adapter 变换后的特征，形状与输入相同 (batch_size, seq_len, input_size)。
        """
        # 先降维，再非线性激活，再升维；形状始终为 (batch_size, seq_len, input_size)
        return self.up_project(self.nolinearity(self.down_project(x)))


class BertSelfOutput(nn.Module):
    """
    重写版 BertSelfOutput：在原有 dense → Dropout → LayerNorm 流程中插入 Adapter 层。

    原始 BERT 中 BertSelfOutput 负责将自注意力的输出做线性变换 + 残差 + LayerNorm。
    此处在 Dropout 之后、残差连接之前插入 AdapterLayer，使 Adapter 输出参与残差相加。

    Args:
        config      (BertConfig)   : BERT 配置对象，提供 hidden_size、layer_norm_eps、hidden_dropout_prob。
        adapter_size(int)          : Adapter 瓶颈维度。
    """
    def __init__(self, config, adapter_size):
        super().__init__()  # 调用父类初始化

        # 原始线性变换：hidden_size → hidden_size（768→768）
        # 类型: nn.Linear，权重形状: (hidden_size, hidden_size)
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)

        # 原始 LayerNorm：对残差相加后的特征做归一化，稳定训练
        # eps=layer_norm_eps 防止除零，bert-base 默认 1e-12
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

        # 原始 Dropout：在训练时随机置零，防止过拟合
        # p=hidden_dropout_prob，bert-base 默认 0.1
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

        # 插入的 Adapter 瓶颈层：仅训练此层参数，原始 dense/LayerNorm/dropout 权重被冻结
        self.adapter = AdapterLayer(config.hidden_size, adapter_size)

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        """
        前向传播：dense → Dropout → Adapter → LayerNorm(+ 残差)。

        Args:
            hidden_states (torch.Tensor): 自注意力输出，形状 (batch_size, seq_len, hidden_size)。
            input_tensor  (torch.Tensor): 残差连接的输入（注意力层之前的特征），形状相同。

        Returns:
            torch.Tensor: 归一化后的输出，形状 (batch_size, seq_len, hidden_size)。
        """
        hidden_states = self.dense(hidden_states)       # 线性变换：768→768
        hidden_states = self.dropout(hidden_states)     # Dropout：随机置零防过拟合
        hidden_states = self.adapter(hidden_states)     # Adapter 适配：特征经瓶颈变换后返回原维度
        hidden_states = self.LayerNorm(hidden_states + input_tensor)  # 残差 + LayerNorm
        return hidden_states


class BertOutput(nn.Module):
    """
    重写版 BertOutput：在 FFN 输出层插入 Adapter。

    原始 BERT 中 BertOutput 负责将 FFN 中间层（intermediate_size=3072）映射回 hidden_size，
    再做残差 + LayerNorm。此处同样在 Dropout 之后插入 AdapterLayer。

    Args:
        config      (BertConfig): BERT 配置对象，提供 intermediate_size、hidden_size 等参数。
        adapter_size(int)       : Adapter 瓶颈维度。
    """
    def __init__(self, config, adapter_size):
        super().__init__()  # 调用父类初始化

        # 原始线性变换：FFN 中间维度 → hidden_size（3072→768）
        # intermediate_size=3072，hidden_size=768
        self.dense = nn.Linear(config.intermediate_size, config.hidden_size)

        # 原始 LayerNorm：对残差相加后的特征做归一化
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

        # 原始 Dropout：防止过拟合
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

        # 插入的 Adapter 瓶颈层
        self.adapter = AdapterLayer(config.hidden_size, adapter_size)

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        """
        前向传播：dense（3072→768）→ Dropout → Adapter → LayerNorm(+ 残差)。

        Args:
            hidden_states (torch.Tensor): FFN 中间层输出，形状 (batch_size, seq_len, intermediate_size=3072)。
            input_tensor  (torch.Tensor): 残差连接输入，形状 (batch_size, seq_len, hidden_size=768)。

        Returns:
            torch.Tensor: 归一化后的输出，形状 (batch_size, seq_len, hidden_size=768)。
        """
        hidden_states = self.dense(hidden_states)       # 3072 → 768
        hidden_states = self.dropout(hidden_states)     # Dropout
        hidden_states = self.adapter(hidden_states)     # Adapter 适配：768→adapter_size→768
        hidden_states = self.LayerNorm(hidden_states + input_tensor)  # 残差 + LayerNorm
        return hidden_states


class AdapterBert(nn.Module):
    """
    Adapter Tuning BERT：在 BERT 每层的注意力输出层和 FFN 输出层各插入一个 Adapter，
    冻结原始 BERT 所有权重，只训练 Adapter 参数和分类头。

    实现方式：替换每层的 BertSelfOutput 和 BertOutput 为自定义重写版本，
    然后将预训练权重载回（不含 Adapter 部分），最后按名称冻结非 Adapter 参数。

    Args:
        adapter_size (int): Adapter 瓶颈维度，越小参数越少，默认 64。
    """
    def __init__(self, adapter_size=64):
        """
        初始化 AdapterBert：加载 BERT → 替换层 → 载入预训练权重 → 冻结非 Adapter 参数。

        Args:
            adapter_size (int): Adapter 瓶颈维度。
        """
        super().__init__()  # 调用父类初始化

        # 加载预训练 BERT-base-uncased
        self.model = AutoModel.from_pretrained("bert-base-uncased", cache_dir=CACHE_DIR_MODEL/"bert-base-uncased")

        # 提前保存预训练权重（state_dict）的快照，稍后重新载入
        # 注意：此时的 state_dict 不含 Adapter 层的参数（Adapter 层尚未添加）
        # 必须在 _apply_adapter 替换层之前保存，否则替换后层名称已改变，load_state_dict 会因键不匹配而报错
        pretrained_state_dict = self.model.state_dict()  # 类型: OrderedDict[str, Tensor]
        print(self.model.config)  # 打印 BERT 配置，查看 hidden_size、intermediate_size 等超参

        # 线性分类头：768 维 → 1 维 logit
        self.classifier = nn.Linear(self.model.config.hidden_size, 1)

        # 替换每层的 BertSelfOutput 和 BertOutput 为含 Adapter 的重写版本
        # 替换后，模型中 adapter.* 参数名对应 Adapter 层的权重（随机初始化）
        self._apply_adapter(adapter_size=adapter_size)

        # 补全快照中缺失的 Adapter 键：_apply_adapter 替换层后模型新增了 adapter.* 参数，
        # 而原始快照中没有这些键，直接 load_state_dict 会因键名不匹配报 Missing key 错误；
        # 此处将 Adapter 当前值（随机初始化）填入快照，使键集合与模型完全一致后再载入。
        for name, param in self.model.named_parameters():
            if "adapter" in name:
                pretrained_state_dict[name] = param  # 用当前（随机初始化的）Adapter 参数填充快照

        # 将补充了 Adapter 键的 state_dict 载入模型：
        # 原始预训练权重恢复，Adapter 层保留随机初始化值
        self.model.load_state_dict(pretrained_state_dict)

        # 冻结所有非 Adapter 参数（即所有名称不含 "adapter" 的参数）
        # Adapter 参数（requires_grad=True）将是唯一参与训练的 BERT 内部参数
        for name, param in self.model.named_parameters():
            if "adapter" not in name:
                param.requires_grad = False  # 冻结：不计算梯度，不参与更新

        # 删除预训练权重快照，释放内存（该对象已不再需要）
        del pretrained_state_dict

    def _apply_adapter(self, adapter_size):
        """
        替换 BERT 每层的 BertSelfOutput 和 BertOutput 为含 Adapter 的版本。

        Args:
            adapter_size (int): Adapter 瓶颈维度。
        """
        # 遍历 BERT 编码器所有 12 层，逐层替换两个输出子层
        for layer in self.model.encoder.layer:
            # layer.attention.output 原类型: BertSelfOutput（无 Adapter）
            # 替换为自定义 BertSelfOutput（含 Adapter），层名 "attention.output" 保持不变
            layer.attention.output = BertSelfOutput(self.model.config, adapter_size)

            # layer.output 原类型: BertOutput（无 Adapter）
            # 替换为自定义 BertOutput（含 Adapter），层名 "output" 保持不变
            layer.output = BertOutput(self.model.config, adapter_size)

    def forward(self, input_ids, attention_mask, token_type_ids):
        """
        前向传播：通过含 Adapter 的 BERT 提取特征，经分类头输出概率。

        Args:
            input_ids      (torch.Tensor): token id，形状 (batch_size, seq_len)。
            attention_mask (torch.Tensor): 注意力掩码，形状 (batch_size, seq_len)。
            token_type_ids (torch.Tensor): 句子类型 id，形状 (batch_size, seq_len)，单句任务全 0。

        Returns:
            torch.Tensor: 正类概率，形状 (batch_size,)，值域 [0, 1]。
        """
        # BERT 前向传播，Adapter 层在每层注意力输出和 FFN 输出处自动被调用
        # outputs 类型: BaseModelOutputWithPoolingAndCrossAttentions
        outputs = self.model(input_ids, attention_mask, token_type_ids)

        # 取最后一层 [CLS] token 的隐藏状态，形状: (batch_size, hidden_size=768)
        feature = outputs.last_hidden_state[:, 0, :]

        logits = self.classifier(feature)         # 线性映射，形状: (batch_size, 1)
        return torch.sigmoid(logits).squeeze()    # sigmoid + squeeze，形状: (batch_size,)


# 实例化 Adapter Tuning BERT（每个 Adapter 瓶颈维度 64）
adapter_bert = AdapterBert(adapter_size=64)
print('-'*50)
print(adapter_bert)   # 打印模型结构，观察每层 attention.output 和 output 已含 AdapterLayer
print('-'*50)
count_parameters(adapter_bert)  # 打印参数统计，验证只有 Adapter 和分类头参与训练



In [ ]:
# 训练 Adapter Tuning 模型（可训练参数：所有 Adapter 层的 down_project + up_project 权重）
training_record["Adapter Tuning"] = train(
    adapter_bert, train_loader, val_loader, device,
    num_epochs=num_epochs, patience=patience
)

In [ ]:
# 释放 Adapter Tuning 模型占用的 GPU 显存
del adapter_bert

## 八、汇总对比图

In [ ]:
# 绘制所有 PEFT 方法的验证准确率汇总对比图
# 横轴: Epoch，纵轴: 验证集准确率（val_acc）
# 各方法在同一图中用不同颜色折线展示，便于直观比较效果与参数效率的权衡关系
plot_training_record(training_record, metric_name="val_acc")